# 第 78 课：推理服务的可观测性
## Trace、结构化日志、低基数指标、SLO 告警与故障定位

上一课的流式服务已经会幂等重试、断线续传、取消、背压和回收 KV。这一课要回答：**出现拒绝或超时时，怎样找到哪条请求、哪个模型版本、哪一阶段出了问题，同时不把用户文本写入监控？**

本课独立包含一套**真实 CPU 因果模型＋Q/V LoRA＋真实分页 KV 张量**，不是仅用假的日志字典演示。但时间仍是**确定性逻辑 tick**，不能当成 GPU 毫秒延迟。运行模型来自随机初始化，不是预训练模型的质量评测。

产物：请求级 Trace、受限结构化日志、低基数 Counter/Histogram、确定分母的 SLI、双窗口告警、证据驱动的故障钻取和干净重载。它们全部使用标准库＋PyTorch 离线运行，**不是实际 OTLP、Prometheus 或 HTTP 服务端。**

## 0. 运行方式与安全边界

选择已安装 PyTorch 的 Python 3 内核，从头执行 Notebook。全部实现均内嵌，不从配套 `.py` 或上一课导入。默认跑一套完整 CPU 场景并写本地时间戳目录；不下载权重，不访问网络，不启动后台服务。

命令行版本：

```bash
python lesson_78_serving_observability_traces_slo_diagnostics.py --checks-only
python lesson_78_serving_observability_traces_slo_diagnostics.py --demo-only
python lesson_78_serving_observability_traces_slo_diagnostics.py
python lesson_78_serving_observability_traces_slo_diagnostics.py --reload-run lesson_78_outputs/实际目录
```

Notebook 前半部分的基础模型/缓存源码较长，折叠部分只为阅读便利，仍已包含且执行。独立练习的开关**默认关闭**，参考答案放在最后，用不同名称，不覆盖你的代码。

**隐私约定：**遥测不记录原始提示、token ID、principal、幂等键或响应正文。日志关联 ID 可查询本地受信记录，但不能视为防止用户推断的安全凭证；生产环境还需要权限、采样、存储保留与数据治理。

## 1. 系统数据流：三个观测面，职责不同

```text
受信网关（principal只在内部）
    │
    ├─ submit  ──真实增量prefill──真实KV页池
    │                               │
    ├─ tick   ──真实decode───────────┘
    │     └─delta / done / cancelled / deadline / capacity
    │
    └─ 观测包装器
          ├── Trace：每个逻辑操作的root / prefill / decode / release
          ├── Log：受限事件名 + trace_id/span_id + 白名单属性
          └── Metrics：低基数版本/终态 Counter，逻辑tick Histogram
                         ↓
                   服务SLI → 双窗口教学告警 → 有证据的故障线索
```

一次请求可触发多个日志、多个 span，但应该只有**一个逻辑请求 root**和一个终态。重复提交是新的一次 *attempt*，不是新的推理计算。

OpenTelemetry Trace 由 trace ID、父子 span、起止时间与属性构成；日志可以通过 trace/span ID 关联。[OTel Traces](https://opentelemetry.io/docs/concepts/signals/traces/) · [OTel 上下文传播](https://opentelemetry.io/docs/concepts/context-propagation/)

下面的实现借鉴这些概念，但没有 SDK、W3C `traceparent` header、跨进程导出或 collector。不要将教学 JSON 文件叫作可直接摄取的 OTLP。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第78课：推理服务可观测性——Trace、结构化日志、SLO与故障定位。

独立离线PyTorch教学实验；真实微型因果Transformer/LoRA增量KV张量
与完整块前缀缓存；模拟确定性单进程事件循环的流式服务可靠性。
本课使用真实CPU KV与确定性逻辑tick，不宣称真实网络/生产级OTel或Prometheus接入。
"""
from __future__ import annotations

# %% 01 环境、确定性身份与范围
import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter, OrderedDict
import secrets
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装或下载模型。") from exc

VERSION = "lesson78-observability-v1"
SOURCES = {
    "pagedattention": "https://arxiv.org/abs/2309.06180",
    "transformers_cache": "https://huggingface.co/docs/transformers/main/en/cache_explanation",
    "pytorch_attention": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html",
    "pytorch_inference": "https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
    "vllm_prefix_cache": "https://docs.vllm.ai/en/latest/design/prefix_caching/",
    "vllm_security": "https://docs.vllm.ai/en/stable/usage/security/",
    "vllm_internals": "https://vllm.ai/blog/2025-09-05-anatomy-of-vllm",
    "vllm_metrics": "https://github.com/vllm-project/vllm/blob/main/docs/design/metrics.md",
    "vllm_benchmark": "https://docs.vllm.ai/en/latest/benchmarking/cli/",
    "vllm_request_metrics": "https://docs.vllm.ai/en/latest/features/per_request_metrics/",
    "triton_batcher": "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html",
    "sre_overload": "https://sre.google/sre-book/handling-overload/",
    "sse": "https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events",
    "sse_standard": "https://html.spec.whatwg.org/multipage/server-sent-events.html",
    "aws_idempotency": "https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/",
    "ietf_idempotency_draft": "https://datatracker.ietf.org/doc/html/draft-ietf-httpapi-idempotency-key-header-08",
    "otel_traces": "https://opentelemetry.io/docs/concepts/signals/traces/",
    "otel_context": "https://opentelemetry.io/docs/concepts/context-propagation/",
    "prometheus_cardinality": "https://prometheus.io/docs/practices/naming/",
    "sre_alerting": "https://sre.google/workbook/alerting-on-slos/",
    "genai_semconv": "https://opentelemetry.io/docs/specs/semconv/gen-ai/",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(7801)
print('Python:',platform.python_version())
print('PyTorch:',torch.__version__,'CUDA可用:',torch.cuda.is_available())
print('默认运行范围：真实CPU KV / 逻辑tick / 自定义观测schema')

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA可用: False
默认运行范围：真实CPU KV / 逻辑tick / 自定义观测schema


## 2. 真实模型与 tokenizer：不是虚构生成服务

复用上一课的实体但在本文件中**完整重写并保留源码**：教学 UTF-8 字节合并 tokenizer、Chat 模板、两层因果 Transformer、两个不同 LoRA 版本，以及每个 token 真正读写 K/V 的分页页池。

- prefill 用当前冻结模型的真实权重计算 Q/K/V。
- decode 先提交新的 KV，再发布 delta；前缀复用受模型、模板与租户可信作用域限制。
- 每一步增量输出与完整上下文前向核对；出错会抛异常，不会伪造“模型成功”。
- 两份 LoRA 版本 `v0001/v0002` 有不同权重摘要。Trace 记录 model fingerprint，指标只记录有界版本名，避免按整个哈希构造指标 series。

随机小模型的 token 输出本身没有语义价值，本课分析的是**推理设施的可观测性**，不是回答质量。

In [3]:
# %% 02 固定教学分词器与真实因果/LoRA模型（独立于前课文件）
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
# %% 03 同基座不同Adapter的只读模型版本
def build_models(seed: int=7201):
    """同一随机基座 + 两份独立Q/V LoRA。版本只是不可变教学对象。"""
    require(type(seed) is int and seed >= 0, 'seed非法')
    seed_all(seed)
    tok = ByteMergeChatTokenizer()
    cfg = ToyConfig(tok.vocab_size, dim=32, heads=4, layers=2, ff_dim=64, max_length=192)
    base = TinyChatLM(cfg)
    a = inject_lora(base, rank=4, alpha=8.0)
    b = inject_lora(base, rank=4, alpha=8.0)
    # B=0时LoRA作用为零；通过不同adapter B制造第二个可核对的版本。
    generator = torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for _,param in adapters(b):
            if param is not None and param.ndim == 2 and param.shape[1] == 4:  # B:[out,rank]
                param.copy_(torch.randn(param.shape, generator=generator)*0.12)
    models={'v0001':a.requires_grad_(False).eval(),
            'v0002':b.requires_grad_(False).eval()}
    require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
            '服务模型必须全部冻结')
    require(base_hash(models['v0001']) == base_hash(models['v0002']),
            '两个版本必须来自同一基座')
    require(state_hash(models['v0001'].state_dict()) != state_hash(models['v0002'].state_dict()),
            '版本权重应该不同')
    return tok,models

In [5]:
# %% 04 实体KV页池：容量、块表、引用计数与回滚
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

In [6]:
# %% 05 增量模型执行器：Q/K/V真实写入、块表聚合与全上下文比较
class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [7]:
# %% 06 可信共享范围、模型/模板/媒体身份与完整块哈希链
@dataclass(frozen=True)
class CacheRequest:
    """教学内部调用：principal必须由可信网关鉴权后传入，不能让客户端自行声明。"""
    sid: str
    principal: str
    version: str
    messages: tuple[tuple[str,str], ...]
    template_revision: str = "chat-v1"
    media_bytes: bytes | None = None
    add_generation_prompt: bool = True


class TrustedScopeAuthority:
    """服务器控制隔离组与加密随机盐，不接受请求自行提供salt。

    本地示例没有HTTP/登录鉴权。真实调用者必须由网关先确认principal身份，
    本类不防御攻击者能直接调用内部Python对象的情形。
    """
    def __init__(self, principal_groups: dict[str,str]):
        require(type(principal_groups) is dict and bool(principal_groups), '需要非空可信身份组')
        require(all(type(k) is str and k and type(v) is str and v
                    for k,v in principal_groups.items()),'可信身份/共享组非法')
        self._groups=principal_groups.copy()
        self._secrets={g:secrets.token_bytes(32) for g in set(principal_groups.values())}

    def group(self,principal: str) -> str:
        require(type(principal) is str and principal in self._groups,
                '未知或未认证主体不能使用Prefix Cache')
        return self._groups[principal]

    def fingerprint(self,principal: str) -> str:
        group=self.group(principal)
        return hashlib.sha256(self._secrets[group]).hexdigest()

    def rotate(self,group: str) -> str:
        require(group in self._secrets, '未知共享组')
        old=hashlib.sha256(self._secrets[group]).hexdigest()
        self._secrets[group]=secrets.token_bytes(32)
        require(self.fingerprint(next(k for k,v in self._groups.items() if v==group))!=old,
                '加密随机盐轮换未生效')
        return old

    def shared(self,p1:str,p2:str)->bool:
        return self.group(p1)==self.group(p2)


@dataclass(frozen=True)
class CacheIdentity:
    version: str
    model_hash: str
    base_hash: str
    tokenizer_hash: str
    template_hash: str
    scope_hash: str
    media_hash: str
    compute_hash: str


def request_tokens(tok:ByteMergeChatTokenizer,req:CacheRequest)->list[int]:
    require(isinstance(req,CacheRequest),'必须传入CacheRequest')
    require(type(req.messages) is tuple and bool(req.messages),'消息必须为非空tuple')
    require(type(req.add_generation_prompt) is bool,'生成提示选项非法')
    bounded_str(req.template_revision,64,'模板修订号')
    require(bool(req.template_revision),'模板修订号不能为空')
    messages=[]
    for item in req.messages:
        require(type(item) is tuple and len(item)==2 and item[0] in tok.role_ids
                and type(item[1]) is str,'消息必须为(role,content)')
        messages.append({'role':item[0],'content':item[1]})
    return tok.apply_chat_template(messages,add_generation_prompt=req.add_generation_prompt)


def make_identity(req:CacheRequest,authority:TrustedScopeAuthority,tok:ByteMergeChatTokenizer,
                  models:dict[str,TinyChatLM],saved_hashes:dict[str,str]) -> CacheIdentity:
    require(req.version in models and req.version in saved_hashes,'版本没有被可信模型库登记')
    authority.group(req.principal)
    require(type(req.media_bytes) in (bytes,type(None)),'媒体身份必须取自受信预处理的原始字节')
    require(req.media_bytes is None or len(req.media_bytes)<=4096,
            '教学元数据媒体上限4096字节；不允许超大请求键')
    # 禁止在同一个版本名称下原地修改模型而继续读取旧KV。
    model=models[req.version]
    current=state_hash(model.state_dict())
    require(current==saved_hashes[req.version], '热更新必须显式替换并失效缓存，不能原地修改模型')
    return CacheIdentity(
        version=req.version,
        model_hash=current,
        base_hash=base_hash(model),
        tokenizer_hash=digest(tok.contract()),
        template_hash=digest({'family':'toy-message-boundary','revision':req.template_revision,
                              'generation_prompt':req.add_generation_prompt,
                              'role_end':'assistant=eos,others=im_end',
                              'tokenizer':tok.contract()}),
        scope_hash=authority.fingerprint(req.principal),
        media_hash=('none' if req.media_bytes is None else
                    hashlib.sha256(req.media_bytes).hexdigest()),
        compute_hash=digest({'cfg':asdict(model.cfg),'dtype':str(next(model.parameters()).dtype),
                             'position':'absolute','attention':'full causal',
                             'kv_layout':'blocks,layer,kv,slot,heads,dim',
                             'runtime':'toy-cpu-fp32-v1'}),
    )


def prefix_keys(identity:CacheIdentity,ids:list[int],block_size:int)->list[str]:
    require(isinstance(identity,CacheIdentity),'缺少可信缓存身份')
    require(type(block_size) is int and block_size>0,'block_size必须为正整数')
    require(type(ids) is list and all(type(t) is int and t>=0 for t in ids),'token列表非法')
    parent=digest({'kind':'scoped-prefix-root','version':VERSION,'identity':asdict(identity)})
    keys=[]
    for start in range(0,len(ids)//block_size*block_size,block_size):
        block=ids[start:start+block_size]
        parent=digest({'kind':'full-prefix-block','parent_hash':parent,
                       'position':start,'token_ids':block})
        keys.append(parent)
    return keys


@dataclass
class CachedBlock:
    key: str
    physical_page: int
    identity: CacheIdentity
    prefix_tokens: tuple[int,...]   # 仅在可信进程内保留，用于哈希碰撞检测；报告禁止展示
    block_tokens: tuple[int,...]
    parent_key: str | None
    end_logits: torch.Tensor
    kv_hash: str
    logits_hash: str
    last_access_tick: int

In [8]:
# %% 07 对真实KV页加缓存所有权：命中、LRU、失效、物理回收
class PrefixKVPool(PagedKVPool):
    """缓存pin与活动会话引用共同拥有物理页。只发布已提交的完整块。

    与工业vLLM不同：CPU同步、预分配Tensor、无真实异步锁、无融合页注意力核。
    精确前缀tokens用于碰撞检测，存储成本未计入KV字节预算。
    """
    def __init__(self,capacity_blocks:int,block_size:int,model_cfg:ToyConfig,versions:set[str],
                 cache_capacity_blocks:int|None=None):
        super().__init__(capacity_blocks,block_size,model_cfg,versions)
        cache_capacity_blocks=capacity_blocks if cache_capacity_blocks is None else cache_capacity_blocks
        require(type(cache_capacity_blocks) is int and 0<=cache_capacity_blocks<=capacity_blocks,
                '缓存块上限不能超过物理容量')
        self.cache_capacity_blocks=cache_capacity_blocks
        self.cache:dict[str,CachedBlock]={}
        self.clock=0
        self.hit_blocks=0;self.hit_tokens=0;self.miss_requests=0
        self.promotions=0;self.evictions=0;self.invalidations=0
        self.skipped_promotions=0;self.zero_hit_requests=0

    def _tick(self):
        self.clock+=1
        return self.clock

    def _page_hash(self,bid:int)->str:
        return state_hash({'physical_KV':self.data[bid]})

    def _validate_entry(self,e:CachedBlock,identity:CacheIdentity,desired_prefix:tuple[int,...],
                        requested_key:str):
        require(e.key==requested_key and e.identity==identity and
                e.prefix_tokens==desired_prefix,'哈希碰撞或缓存身份/前缀不一致，必须拒绝命中')
        require(e.block_tokens==desired_prefix[-self.block_size:]
                and len(e.block_tokens)==self.block_size,'只能命中完整块')
        require(e.kv_hash==self._page_hash(e.physical_page),'缓存KV字节损坏，拒绝复用')
        require(e.logits_hash==state_hash({'end_logits':e.end_logits}),
                '缓存块末端logits字节损坏，拒绝复用')
        require(self.refs[e.physical_page]>=1,'缓存页已经释放，索引悬空')

    def lookup(self,identity:CacheIdentity,ids:list[int])->list[CachedBlock]:
        keys=prefix_keys(identity,ids,self.block_size)
        found=[]
        for i,key in enumerate(keys):
            e=self.cache.get(key)
            if e is None:break
            self._validate_entry(e,identity,tuple(ids[:(i+1)*self.block_size]),key)
            # 顺序访问同时更新LRU；匹配只返回连续前缀，不能跳过中间缺失块。
            e.last_access_tick=self._tick()
            found.append(e)
        return found

    def _evict_one_idle(self)->bool:
        choices=[e for e in self.cache.values() if self.refs[e.physical_page]==1]
        if not choices:return False
        victim=min(choices,key=lambda e:(e.last_access_tick,e.key))
        del self.cache[victim.key]
        self._decref(victim.physical_page)
        self.evictions+=1
        return True

    def _alloc(self):
        if not self.free:
            self._evict_one_idle()
        return super()._alloc()

    def begin(self,sid,token_id,max_length):
        # 父类在发现free为空时会先报OOM；尝试驱逐idle cache才进入它的begin。
        if sid in self.sessions:
            s=self.sessions[sid]
            if s.pending is None and not self.free:
                slot=len(s.tokens)%self.block_size
                needs_page=slot==0 or (slot>0 and self.refs[s.block_table[-1]]>1)
                if needs_page:self._evict_one_idle()
        return super().begin(sid,token_id,max_length)

    def publish(self,sid:str,index:int,key:str,identity:CacheIdentity,
                prefix_tokens:tuple[int,...],end_logits:torch.Tensor,
                parent_key:str|None)->bool:
        require(sid in self.sessions,'发布会话不存在')
        s=self.sessions[sid]
        require(s.pending is None and s.version==identity.version,'不允许发布未提交/跨版本KV')
        require(type(index) is int and 0<=index<len(s.block_table),'发布块下标错误')
        require(len(prefix_tokens)==(index+1)*self.block_size and
                tuple(s.tokens[:len(prefix_tokens)])==prefix_tokens,'前缀token身份错误')
        require(end_logits.shape==(self.cfg.vocab_size,) and end_logits.dtype==self.dtype and
                bool(torch.isfinite(end_logits).all()),'末端logits形状/数值错误')
        chain=prefix_keys(identity,list(prefix_tokens),self.block_size)
        require(chain and chain[-1]==key and (chain[-2] if len(chain)>1 else None)==parent_key,
                '发布哈希链与token前缀不一致')
        if key in self.cache:
            self._validate_entry(self.cache[key],identity,prefix_tokens,key)
            return False
        if self.cache_capacity_blocks==0:
            self.skipped_promotions+=1
            return False
        if len(self.cache)>=self.cache_capacity_blocks and not self._evict_one_idle():
            self.skipped_promotions+=1
            return False
        bid=s.block_table[index]
        require(self.refs[bid]>=1,'发布物理页没有会话所有权')
        self.refs[bid]+=1             # 独立cache pin：请求结束后仍可命中
        logit=end_logits.detach().cpu().contiguous().clone()
        self.cache[key]=CachedBlock(key=key,physical_page=bid,identity=identity,
             prefix_tokens=prefix_tokens,block_tokens=prefix_tokens[-self.block_size:],
             parent_key=parent_key,end_logits=logit,kv_hash=self._page_hash(bid),
             logits_hash=state_hash({'end_logits':logit}),last_access_tick=self._tick())
        self.promotions+=1
        self.audit()
        return True

    def invalidate(self,predicate:Callable[[CachedBlock],bool]|None=None)->int:
        """移除pin。即使仍有活动会话，只有它们的引用会保留KV。"""
        selected=[key for key,e in self.cache.items() if predicate is None or predicate(e)]
        for key in selected:
            e=self.cache.pop(key)
            self._decref(e.physical_page)
        self.invalidations+=len(selected)
        self.audit()
        return len(selected)

    def advance_clock_and_expire(self,ticks:int,max_idle_ticks:int)->int:
        require(type(ticks) is int and ticks>=0 and type(max_idle_ticks) is int and max_idle_ticks>=0,
                '逻辑时间参数非法')
        self.clock+=ticks
        return self.invalidate(lambda e: self.clock-e.last_access_tick>max_idle_ticks)

    def cache_usage(self)->dict:
        total=self.usage()
        idle_cached=sum(self.refs[e.physical_page]==1 for e in self.cache.values())
        return {'cache_blocks':len(self.cache),'cache_capacity_blocks':self.cache_capacity_blocks,
                'idle_cached_blocks':idle_cached,'active_cached_blocks':len(self.cache)-idle_cached,
                'cache_pins':len(self.cache),'cache_held_bytes':len(self.cache)*self.bytes_per_block,
                'hit_blocks':self.hit_blocks,'hit_tokens':self.hit_tokens,
                'promotions':self.promotions,'evictions':self.evictions,
                'invalidations':self.invalidations,'skipped_promotions':self.skipped_promotions,
                'live_blocks':total['live_blocks'],'free_blocks':total['free_blocks'],
                'session_count':len(self.sessions),'physical_storage_bytes':total['physical_storage_bytes']}

    def usage(self)->dict:
        result=super().usage()
        result['cache_pins']=len(self.cache)
        result['cache_held_bytes']=len(self.cache)*self.bytes_per_block
        # 物理页可以只由缓存持有，因此旧课的logical_refs-live会变负：需把pin计入。
        result['saved_physical_blocks_by_sharing']=(result['logical_page_references']+
                                                     len(self.cache)-result['live_blocks'])
        return result

    def audit(self)->dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话模型版本不在白名单')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '会话页数与token长度不匹配')
            require(len(s.block_table)==len(set(s.block_table)),'同一会话重复引用页')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'页ID越界')
                expected[bid]+=1
        cache_pages=set()
        for key,e in self.cache.items():
            require(key==e.key and e.physical_page not in cache_pages,'重复pin或缓存键/页错配')
            cache_pages.add(e.physical_page)
            require(0<=e.physical_page<self.capacity_blocks,'缓存物理页非法')
            require(len(e.prefix_tokens)%self.block_size==0 and len(e.block_tokens)==self.block_size,
                    '缓存只允许完整块')
            expected[e.physical_page]+=1
        require(expected==self.refs,'会话引用+缓存pin不等于实际物理引用数')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free-list不守恒')
        require(len(set(self.free))==len(self.free),'free-list有重复')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),'空闲页未清零')
        u=self.usage()
        require(u['saved_physical_blocks_by_sharing']>=0,'物理共享账本负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '字节账本不守恒')
        return u

In [9]:
# %% 08 前缀命中后继续真实预填充，并与完整因果前向核对
class PrefixCacheRunner:
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],
                 pool:PrefixKVPool,authority:TrustedScopeAuthority):
        require(isinstance(pool,PrefixKVPool),'需要有缓存所有权的KV页池')
        self.tok=tok;self.models=models;self.pool=pool;self.authority=authority
        self.runner=CachedModelRunner(tok,models,pool)
        self.bound:dict[str,CacheIdentity]={}
        self.events:list[dict]=[]

    def admit(self,req:CacheRequest)->dict:
        require(isinstance(req,CacheRequest),'请求类型非法')
        self.pool._sid(req.sid)
        require(req.sid not in self.pool.sessions,'sid不可重复入场')
        # 注意：principal须来自已鉴权的内部调用上下文，不应接受HTTP客户端自报。
        ids=request_tokens(self.tok,req)
        require(bool(ids) and len(ids)<=self.pool.cfg.max_length,'提示过长或为空')
        ident=make_identity(req,self.authority,self.tok,self.models,self.runner.model_hashes)
        hits=self.pool.lookup(ident,ids)
        hit_count=len(hits)*self.pool.block_size
        keys=prefix_keys(ident,ids,self.pool.block_size)
        self.pool.open(req.sid,req.version)
        self.bound[req.sid]=ident
        end_logits:dict[int,torch.Tensor]={}
        try:
            state=self.pool.sessions[req.sid]
            for i,e in enumerate(hits):
                self.pool.refs[e.physical_page]+=1
                state.block_table.append(e.physical_page)
                state.tokens.extend(e.block_tokens)
                end_logits[i]=e.end_logits.detach().clone()
            if hits:
                self.runner.last_logits[req.sid]=hits[-1].end_logits.detach().clone()
            self.pool.audit()
            for t in ids[hit_count:]:
                out=self.runner.step(req.sid,t)
                n=len(self.pool.sessions[req.sid].tokens)
                if n%self.pool.block_size==0:
                    end_logits[n//self.pool.block_size-1]=out.detach().clone()
            require(self.pool.sessions[req.sid].tokens==ids,'前缀命中恢复后的token序列错误')
            require(req.sid in self.runner.last_logits,'请求没有任何合法预测logits')
            # 先计算整个prompt，再发布完整块；失败入场不残留不完整页索引。
            parent=None
            for i,key in enumerate(keys):
                prefix=tuple(ids[:(i+1)*self.pool.block_size])
                require(i in end_logits,'完整块末端logits没有计算/回填')
                self.pool.publish(req.sid,i,key,ident,prefix,end_logits[i],parent)
                parent=key
            self.pool.hit_blocks+=len(hits);self.pool.hit_tokens+=hit_count
            if not hits:self.pool.zero_hit_requests+=1
            max_delta=self.runner.assert_consistent(req.sid)
            result={'sid':req.sid,'version':req.version,'prompt_tokens':len(ids),
                    'hit_blocks':len(hits),'hit_tokens':hit_count,'new_tokens_computed':len(ids)-hit_count,
                    'max_logits_abs_delta':max_delta,'cache_blocks_after':len(self.pool.cache),
                    'scope':'internal_authenticated_principal'}
            self.events.append(result.copy())
            return result
        except Exception:
            # 失败不留下活动会话/泄漏物理页；不回滚之前已发生的idle LRU驱逐。
            if req.sid in self.pool.sessions:
                self.runner.close(req.sid)
            self.bound.pop(req.sid,None)
            self.pool.audit()
            raise

    def continue_token(self,sid:str,token_id:int)->float:
        require(sid in self.bound and sid in self.pool.sessions,'不存在的会话')
        identity=self.bound[sid]
        require(identity.model_hash==state_hash(self.models[identity.version].state_dict()),
                '活跃会话期间模型权重不可原地热修改')
        self.runner.step(sid,token_id)
        return self.runner.assert_consistent(sid)

    def fork_session(self,parent:str,child:str,principal:str):
        require(parent in self.bound and parent in self.pool.sessions,'父会话不存在')
        require(child not in self.bound and child not in self.pool.sessions,'子会话已经存在')
        identity=self.bound[parent]
        require(self.authority.fingerprint(principal)==identity.scope_hash,
                'fork只允许相同可信缓存共享范围，不能跨租户')
        self.runner.fork(parent,child,identity.version)
        self.bound[child]=identity
        self.pool.audit()

    def close(self,sid:str):
        require(sid in self.bound,'未授权/已经关闭的会话')
        self.runner.close(sid)
        self.bound.pop(sid)
        self.pool.audit()

    def invalidate_version(self,version:str)->int:
        require(version in self.models,'未知版本')
        return self.pool.invalidate(lambda e:e.identity.version==version)

    def rotate_group(self,group:str)->int:
        old_fp=self.authority.rotate(group)
        return self.pool.invalidate(lambda e:e.identity.scope_hash==old_fp)

    def replace_version(self,version:str,new_model:TinyChatLM)->int:
        require(version in self.models,'未知版本不能替换')
        require(all(s.version!=version for s in self.pool.sessions.values()),
                '有该版本的活跃会话，禁止原地替换/撕裂读写')
        require(isinstance(new_model,TinyChatLM) and new_model.cfg==self.pool.cfg,
                '新模型配置或类型错误')
        require(next(new_model.parameters()).device.type=='cpu'
                and next(new_model.parameters()).dtype==torch.float32,
                '新模型必须是CPU FP32')
        require(all(not p.requires_grad for p in new_model.parameters()),'新模型必须全部冻结')
        replacement=copy.deepcopy(new_model).eval()
        invalidated=self.invalidate_version(version)
        self.models[version]=replacement
        self.runner.models[version]=replacement
        self.runner.model_hashes[version]=state_hash(replacement.state_dict())
        return invalidated

    def final_flush(self)->dict:
        require(not self.pool.sessions and not self.bound,'flush前必须结束全部会话')
        self.pool.invalidate()
        usage=self.pool.audit()
        require(usage['live_blocks']==0 and not bool(self.pool.data.count_nonzero()),
                '全部缓存清空后仍有KV残留')
        return usage

### 先验证一次真实 KV 增量计算

下面调用的 `assert_consistent` 会用同一权重重新运行完整上下文前向，再与真实 KV 增量输出比较。它不是靠预设一个期望 logit 数组来假装“模型正确”。

In [10]:
# %% 09 推理请求契约：租户作用域、幂等指纹和确定性时钟
@dataclass(frozen=True)
class StreamRequest:
    """principal由可信网关注入，不可直接使用未鉴权HTTP字段。"""
    principal: str
    idempotency_key: str
    version: str
    text: str
    timeout_ticks: int = 12
    max_new_tokens: int = 5
    template_revision: str = "chat-v1"

    def validate(self):
        for name in ("principal", "idempotency_key", "version", "template_revision"):
            val=getattr(self,name)
            bounded_str(val,64,name)
            require(bool(re.fullmatch(r"[A-Za-z0-9_-]+",val)),name+"只允许安全字符")
        bounded_str(self.text,256,"text")
        require(0 < len(self.text.encode('utf-8')) <= 256,'空输入或超长输入')
        require(type(self.timeout_ticks) is int and 1 <= self.timeout_ticks <= 1000,'timeout_ticks非法')
        require(type(self.max_new_tokens) is int and 1 <= self.max_new_tokens <= 12,'max_new_tokens非法')

    def fingerprint(self):
        self.validate()
        return digest(asdict(self))


@dataclass(frozen=True)
class StreamConfig:
    capacity_blocks: int = 96
    block_size: int = 4
    cache_capacity_blocks: int = 12
    max_sessions: int = 4
    max_records: int = 80
    max_unacked: int = 3
    slow_consumer_ticks: int = 8
    retention_ticks: int = 12

    def validate(self):
        for key,value in asdict(self).items():
            require(type(value) is int and value > 0,key+'必须为正整数')
        require(self.cache_capacity_blocks <= self.capacity_blocks,'cache超过物理页容量')
        require(self.max_unacked >= 2,'至少允许accepted及一次delta')
        require(self.retention_ticks >= 2,'保留期过短')


@dataclass
class StreamRecord:
    sid: str
    request: StreamRequest|None
    fingerprint: str
    identity_hash: str
    accepted_tick: int
    deadline_tick: int
    state: str = 'active'
    event_log: list[dict] = field(default_factory=list)
    acked_seq: int = -1
    last_ack_tick: int = 0
    output_ids: list[int] = field(default_factory=list)
    stop_reason: str|None = None
    end_tick: int|None = None
    prefill: dict|None = None
    playback_expired: bool = False


class RejectedStream(LessonError):
    """有明确拒绝原因的服务契约错误。"""


class ReplayGone(RejectedStream):
    """幂等墓碑还在，但重放窗口已过；绝不默默重新生成。"""


class StreamGateway:
    """单进程同步状态机。每一tick推进确定性顺序，不提供网络/事务/线程安全保证。

    关键约定：先完成真实KV写入，再提交delta事件；每个幂等key唯一活跃计算。
    客户端可能重复收到同一事件，用event id去重；不承诺exactly-once delivery。
    accepted事件占一个未ack槽；terminal可越过背压确保可靠结束信号。
    """
    terminal_states={'done','cancelled','deadline_exceeded','slow_consumer','kv_oom','capacity_rejected'}
    valid_events={'accepted','delta','done','cancelled','deadline_exceeded',
                  'slow_consumer','kv_oom','capacity_rejected'}

    def __init__(self,tok,models,authority,config:StreamConfig=StreamConfig()):
        config.validate()
        require(type(models) is dict and set(models)=={'v0001','v0002'},'教学必须使用两个固定模型版本')
        self.tok=tok;self.models=models;self.authority=authority;self.config=config
        cfg=next(iter(models.values())).cfg
        self.pool=PrefixKVPool(config.capacity_blocks,config.block_size,cfg,set(models),
                               cache_capacity_blocks=config.cache_capacity_blocks)
        self.runner=PrefixCacheRunner(tok,models,self.pool,authority)
        self.now=0
        self.records:dict[tuple[str,str],StreamRecord]={}
        self.sid_index:dict[str,StreamRecord]={}
        self.admissions=0;self.duplicate_submissions=0;self.rejected_overload=0
        self.total_tokens=0;self.backpressure_skips=0;self.deadline_count=0
        self.cancel_count=0;self.expiration_count=0;self.slow_count=0
        self.failures=0;self.max_pending_observed=0
        self.audit_log:list[dict]=[]

    def _id_key(self,principal,key):
        require(type(principal) is str and type(key) is str,'租户/幂等键必须为字符串')
        bounded_str(principal,64,'principal');bounded_str(key,64,'idempotency_key')
        self.authority.group(principal)  # 来自服务端可信映射
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',key)),'幂等键非法')
        return principal,key

    def _record(self,principal,key):
        k=self._id_key(principal,key)
        require(k in self.records,'未找到幂等操作；不能凭sid读取其他租户')
        return self.records[k]

    def _emit(self,record:StreamRecord,kind:str,data:dict):
        require(kind in self.valid_events,'未知事件类型')
        require(type(data) is dict and all(type(k) is str for k in data),'事件内容非法')
        require(not record.playback_expired,'重放期已过不能继续追加')
        seq=len(record.event_log)
        event={'id':f'{record.sid}:{seq}','seq':seq,'type':kind,'tick':self.now,
               'data':copy.deepcopy(data)}
        record.event_log.append(event)
        self.max_pending_observed=max(self.max_pending_observed,seq-record.acked_seq)
        return event

    def _active(self):
        return [r for r in self.records.values() if r.state=='active']

    def submit(self,request:StreamRequest):
        require(isinstance(request,StreamRequest),'需要StreamRequest')
        request.validate()
        key=self._id_key(request.principal,request.idempotency_key)
        fingerprint=request.fingerprint()
        if key in self.records:
            old=self.records[key]
            require(old.fingerprint==fingerprint,'同一租户同一幂等键不能修改语义参数')
            self.duplicate_submissions+=1
            return {'status':'expired' if old.playback_expired else 'reused',
                    'sid':old.sid,'state':old.state,'deadline_tick':old.deadline_tick,
                    'new_compute':False}
        require(len(self.records)<self.config.max_records,'幂等墓碑容量耗尽；拒绝而不是静默逐出')
        require(request.version in self.models,'不支持的模型版本')
        # 固定长度预算：拒绝上下文越界，不在计算之后悄悄截断。
        cache_req=CacheRequest(sid='unused',principal=request.principal,version=request.version,
            messages=(('user',request.text),),template_revision=request.template_revision)
        tokens=request_tokens(self.tok,cache_req)
        require(len(tokens)+request.max_new_tokens<=self.pool.cfg.max_length,'prompt+生成预算超过上下文')
        sid=f'rs{len(self.records):04d}'
        record=StreamRecord(sid=sid,request=request,fingerprint=fingerprint,
            identity_hash=digest({'fp':fingerprint,'model':self.runner.runner.model_hashes[request.version],
                                  'tokenizer':self.tok.contract()}),accepted_tick=self.now,
            deadline_tick=self.now+request.timeout_ticks,last_ack_tick=self.now)
        # 先占有幂等位置，然后尝试真实KV预填充；拒绝也保留terminal墓碑。
        self.records[key]=record;self.sid_index[sid]=record
        if len(self._active())>self.config.max_sessions:
            self.rejected_overload+=1
            self._terminate(record,'capacity_rejected',{'reason':'concurrent_session_limit'})
            return {'status':'capacity_rejected','sid':sid,'state':record.state,
                    'deadline_tick':record.deadline_tick,'new_compute':False}
        internal=CacheRequest(sid=sid,principal=request.principal,version=request.version,
                messages=(('user',request.text),),template_revision=request.template_revision)
        try:
            admitted=self.runner.admit(internal)
        except NoKVSpace:
            self.rejected_overload+=1
            self._terminate(record,'capacity_rejected',{'reason':'KV_capacity'})
            return {'status':'capacity_rejected','sid':sid,'state':record.state,
                    'deadline_tick':record.deadline_tick,'new_compute':False}
        except Exception:
            # 编程错误不能伪装为正常终态；清除尚未入场的墓碑并重新抛出。
            self.records.pop(key,None);self.sid_index.pop(sid,None)
            raise
        record.prefill=admitted
        self.admissions+=1
        self._emit(record,'accepted',{'version':request.version,'max_new_tokens':request.max_new_tokens,
                                     'deadline_tick':record.deadline_tick})
        self.audit_log.append({'tick':self.now,'op':'admit','sid':sid,'version':request.version})
        return {'status':'created','sid':sid,'state':record.state,
                'deadline_tick':record.deadline_tick,'new_compute':True}

    def _terminate(self,record:StreamRecord,kind:str,data:dict):
        require(kind in self.terminal_states,'终态类型不合法')
        require(record.state=='active','只能从active进入唯一终态')
        record.state=kind;record.stop_reason=kind;record.end_tick=self.now
        # 终态先释放资源再提交唯一终态事件，避免发了done却没回收KV。
        if record.sid in self.runner.bound:
            self.runner.close(record.sid)
        self._emit(record,kind,{**data,'generated_tokens':len(record.output_ids)})
        self.audit_log.append({'tick':self.now,'op':kind,'sid':record.sid})
        if kind=='cancelled':self.cancel_count+=1
        if kind=='deadline_exceeded':self.deadline_count+=1
        if kind=='slow_consumer':self.slow_count+=1
        if kind=='kv_oom':self.failures+=1

    def cancel(self,principal,key):
        r=self._record(principal,key)
        if r.state=='active':
            self._terminate(r,'cancelled',{'reason':'explicit_request'})
            return {'status':'cancelled','cancelled_now':True}
        return {'status':r.state,'cancelled_now':False}

    def _generate_one(self,record:StreamRecord):
        require(record.state=='active','只能推进活跃请求')
        req=record.request;sid=record.sid
        logits=self.runner.runner.last_logits[sid]
        # 教学贪心生成；固定模型/版本、同样的logits决定token，不使用未声明采样温度。
        next_id=int(logits.argmax())
        try:
            self.runner.continue_token(sid,next_id)  # 先真实KV提交
        except NoKVSpace:
            self._terminate(record,'kv_oom',{'reason':'physical_KV_pool_full'})
            return
        record.output_ids.append(next_id)
        self.total_tokens+=1
        if next_id==self.tok.eos_token_id:
            self._terminate(record,'done',{'finish_reason':'eos'})
        else:
            self._emit(record,'delta',{'token_id':next_id,'position':len(record.output_ids)-1})
            if len(record.output_ids)>=req.max_new_tokens:
                self._terminate(record,'done',{'finish_reason':'length'})

    def tick(self,steps=1):
        require(type(steps) is int and 1<=steps<=1000,'tick步数非法')
        for _ in range(steps):
            self.now+=1
            for r in list(self._active()):
                if self.now>=r.deadline_tick:
                    self._terminate(r,'deadline_exceeded',{'reason':'absolute_request_deadline'})
                    continue
                pending=len(r.event_log)-1-r.acked_seq
                if pending>=self.config.max_unacked:
                    self.backpressure_skips+=1
                    if self.now-r.last_ack_tick>=self.config.slow_consumer_ticks:
                        self._terminate(r,'slow_consumer',{'reason':'unacked_window_timeout'})
                    continue
                self._generate_one(r)
            self.prune()
            self.audit()

    @staticmethod
    def render_sse(event):
        require(type(event) is dict and set(event)=={'id','seq','type','tick','data'},'事件结构非法')
        require(type(event['id']) is str and bool(re.fullmatch(r'rs\d{4}:\d+',event['id'])),
                'SSE id非法')
        require(event['type'] in StreamGateway.valid_events,'事件类型非法')
        # 所有data均以单行JSON编码，避免把用户输入作为SSE控制行。
        data=canonical({'tick':event['tick'],**event['data']})
        return f"id: {event['id']}\nevent: {event['type']}\ndata: {data}\n\n"

    @staticmethod
    def parse_sse_frame(frame):
        require(type(frame) is str and frame.endswith('\n\n'),'单帧SSE必须以空行结束')
        parts=frame[:-2].split('\n')
        require(len(parts)==3,'仅接受id/event/data三行教学帧')
        keys=['id: ','event: ','data: ']
        require(all(line.startswith(prefix) for line,prefix in zip(parts,keys)),'SSE字段或顺序非法')
        event_id,typ,raw=[line[len(prefix):] for line,prefix in zip(parts,keys)]
        require(bool(re.fullmatch(r'rs\d{4}:\d+',event_id)) and typ in StreamGateway.valid_events,
                'SSE id/type非法')
        payload=json.loads(raw)
        require(type(payload) is dict and type(payload.get('tick')) is int,'数据体非法')
        return {'id':event_id,'type':typ,'data':payload}

    def _cursor(self,record,last_event_id):
        if last_event_id is None:return -1
        bounded_str(last_event_id,64,'Last-Event-ID')
        require(last_event_id.startswith(record.sid+':'),'事件ID不属于本操作/本租户')
        tail=last_event_id[len(record.sid)+1:]
        require(bool(re.fullmatch(r'(0|[1-9]\d*)',tail)),'事件序号非法')
        cursor=int(tail)
        require(cursor<len(record.event_log),'客户端不能声称未来事件或已丢失事件')
        return cursor

    def poll(self,principal,key,last_event_id=None):
        r=self._record(principal,key)
        if r.playback_expired:raise ReplayGone('事件保留期已结束；墓碑仍阻止隐式重跑')
        cursor=self._cursor(r,last_event_id)
        return [copy.deepcopy(e) for e in r.event_log if e['seq']>cursor]

    def ack(self,principal,key,last_event_id):
        r=self._record(principal,key)
        if r.playback_expired:raise ReplayGone('事件已经过期')
        cursor=self._cursor(r,last_event_id)
        require(cursor>=r.acked_seq,'ack不能倒退')
        # 同序号重复ack为幂等操作，时间不刷新，避免无限延长慢客户端租约。
        changed=cursor>r.acked_seq
        if changed:
            r.acked_seq=cursor;r.last_ack_tick=self.now
        return {'acked_seq':r.acked_seq,'new_ack':changed,
                'pending':len(r.event_log)-1-r.acked_seq}

    def prune(self):
        for r in self.records.values():
            if r.state!='active' and not r.playback_expired and r.end_tick is not None:
                if self.now-r.end_tick>=self.config.retention_ticks:
                    r.event_log=[];r.output_ids=[];r.prefill=None;r.request=None
                    r.playback_expired=True;self.expiration_count+=1

    def audit(self):
        usage=self.pool.audit()
        active=self._active()
        require(len(active)<=self.config.max_sessions,'活动会话超过准入上限')
        require(len(self.records)<=self.config.max_records,'墓碑记录上限不守恒')
        for r in self.records.values():
            if not r.playback_expired:
                require(r.request is not None and r.fingerprint==r.request.fingerprint(),
                        '操作指纹与请求内容改变')
            if r.state=='active':
                require(r.sid in self.pool.sessions and r.sid in self.runner.bound,
                        '活跃记录缺KV会话/runner绑定')
                require(not r.playback_expired and r.end_tick is None,'活动记录不能过期或结束')
            else:
                require(r.state in self.terminal_states and r.end_tick is not None,'终态非法')
                require(r.sid not in self.pool.sessions and r.sid not in self.runner.bound,
                        '终态有KV会话泄漏')
            if r.playback_expired:
                require(not r.event_log,'过期后不能留下事件数据')
                continue
            require([e['seq'] for e in r.event_log]==list(range(len(r.event_log))),
                    '事件序列必须连续、不能重复提交')
            require(all(e['id']==f'{r.sid}:{e["seq"]}' for e in r.event_log),'事件ID与记录不匹配')
            terminals=[e for e in r.event_log if e['type'] in self.terminal_states]
            require(len(terminals)==(0 if r.state=='active' else 1),'必须恰好一次终态事件')
            require(r.state=='active' or r.event_log[-1]['type']==r.state,'终态必须最后发布')
            require(-1<=r.acked_seq<len(r.event_log),'ack不能越过已提交事件')
            for e in r.event_log:
                require(e['type'] in self.valid_events,'未知事件')
                if e['type']=='delta':
                    require(type(e['data'].get('token_id')) is int,'delta必须包含真实token ID')
            if r.state=='active':
                # accepted计入窗口；终态事件允许多占一个槽确保可见。
                require(len(r.event_log)-1-r.acked_seq<=self.config.max_unacked,
                        '活跃请求超过未确认消息上限')
        require(usage['session_count']==len(active),'活跃记录数与实际KV会话数不一致')
        return {'tick':self.now,'records':len(self.records),'active':len(active),
                'terminal':len(self.records)-len(active),'KV':self.pool.cache_usage(),
                'tokens_generated':self.total_tokens,'duplicate_submissions':self.duplicate_submissions,
                'backpressure_skips':self.backpressure_skips,'max_pending':self.max_pending_observed}

    def final_flush(self):
        for r in list(self._active()):
            self._terminate(r,'cancelled',{'reason':'server_shutdown'})
        usage=self.runner.final_flush()
        require(usage['live_blocks']==0 and usage['session_count']==0 and
                not bool(self.pool.data.count_nonzero()),'最终物理KV仍被持有')
        self.audit()
        return usage

In [11]:
# %% 10 透明客户端：at-least-once续传、事件ID去重与ack
class ReplayClient:
    """单消费方教学模型；真实多客户端ack需要单独的所有权与序列化策略。"""
    def __init__(self,principal,key):
        self.principal=principal;self.key=key
        self.seen:set[str]=set();self.ordered:list[dict]=[]
        self.cursor:str|None=None
        self.duplicates=0

    def receive(self,server:StreamGateway,*,ack=True,disconnect_after=None):
        events=server.poll(self.principal,self.key,self.cursor)
        for event in events:
            if event['id'] in self.seen:
                self.duplicates+=1
                continue
            # 模拟网络只收到前N条：cursor只在交付给本客户端后前进。
            if disconnect_after is not None and len(self.ordered)>=disconnect_after:
                break
            parsed=StreamGateway.parse_sse_frame(StreamGateway.render_sse(event))
            require(parsed['id']==event['id'] and parsed['type']==event['type'],'SSE帧往返不一致')
            self.seen.add(event['id']);self.ordered.append(event)
            self.cursor=event['id']
            if ack:server.ack(self.principal,self.key,self.cursor)
        return len(self.ordered)

    def restart_without_cursor(self):
        # 故意丢失游标但保留去重ID，模拟重复投递。
        self.cursor=None

    @property
    def tokens(self):
        return [e['data']['token_id'] for e in self.ordered if e['type']=='delta']

In [12]:
# %% 11 真实模型与有界场景：幂等、断线、取消、超时、背压与KV容量

def new_service(seed=7701,config:StreamConfig|None=None,model_bundle=None):
    torch.set_num_threads(1)
    tok,models=build_models(seed) if model_bundle is None else model_bundle
    authority=TrustedScopeAuthority({'alice':'tenant-A','bob':'tenant-B','alice2':'tenant-A'})
    return StreamGateway(tok,models,authority,config or StreamConfig())


def safe_request(key,text='hello',principal='alice',version='v0001',timeout=20,max_tokens=5):
    return StreamRequest(principal,key,version,text,timeout,max_tokens)

In [13]:
demo_tok,demo_models = build_models(7801)
demo_svc = new_service(7801,model_bundle=(demo_tok,demo_models))
demo_req = safe_request('probe','hello',max_tokens=3,timeout=15)
demo_response=demo_svc.submit(demo_req)
print('prefill：',demo_svc.sid_index[demo_response['sid']].prefill)
print('完整前向vs真实KV最大logit差：',demo_svc.runner.runner.assert_consistent(demo_response['sid']))
demo_svc.cancel('alice','probe')
print('最终释放KV占用：',demo_svc.final_flush()['live_blocks'])

prefill： {'sid': 'rs0000', 'version': 'v0001', 'prompt_tokens': 7, 'hit_blocks': 0, 'hit_tokens': 0, 'new_tokens_computed': 7, 'max_logits_abs_delta': 3.5762786865234375e-07, 'cache_blocks_after': 1, 'scope': 'internal_authenticated_principal'}
完整前向vs真实KV最大logit差： 3.5762786865234375e-07
最终释放KV占用： 0


## 3. 真实服务契约：不可让遥测改变业务逻辑

请求含受信 principal、幂等键、版本、提示、timeout tick 与最大输出 token 数；服务内部有 accepted/delta/terminal 事件流。主动取消、deadline、背压导致的 slow_consumer 和容量拒绝严格区分。TTL 到期后留下幂等墓碑，不悄悄重计算请求。

本课对这个服务做**观测包装**。包装器不能先写日志就宣布 token 成功，必须在真实 KV 提交完成后才记录 `token_committed`。终态要先释放会话，才记 `kv.release`；输入非法则抛异常，不能变成“用户得了0分”或普通EOS。

`ReplayClient` 可按事件 ID 去重；客户端重放同一事件不增加模型 token Counter。

## 4. Trace：一个逻辑请求的 root，下挂真实阶段

一个请求的示意结构：

```text
inference.request       0──4 tick
  ├─request.admission    0──0
  ├─inference.prefill    0──0（同步调用，耗时未测量）
  ├─stream.poll          0──0
  ├─inference.decode     0──1
  ├─inference.decode     1──2
  └─kv.release          4──4
```

根 `trace_id` 为 32 位十六进制，`span_id` 为16位，子span只存父ID。我们明确使用**逻辑 tick**，所以 `prefill: 0→0` 只代表这一tick发生了 prefill，**不能解释为用时0毫秒**。真实生产应使用单调时钟、性能计时和分布式上下文传播。

每个 root 还记录模型**完整权重摘要**与模板摘要。高基数身份只属于 Trace，不作为低基数指标标签；不要附带原始请求正文。

In [14]:
# %% 12 最小可审计Trace、日志与低基数Metrics（非OTel SDK）
TRACE_ID_RE = re.compile(r'^[0-9a-f]{32}$')
SPAN_ID_RE = re.compile(r'^[0-9a-f]{16}$')
SPAN_NAMES = {'inference.request','request.admission','inference.prefill','inference.decode',
              'request.retry','stream.poll','stream.ack','stream.backpressure','kv.release',
              'stream.cancel'}
SPAN_ATTRIBUTES = {'model_version','input_tokens','cache_hit_tokens','new_prefill_tokens',
                   'new_compute','generated_tokens','response_status','event_count',
                   'model_fingerprint','template_fingerprint',
                   'pending_events','kv_live_after','version','terminal_reason'}
LOG_EVENTS = {'admitted','rejected','retry','token_committed','terminal','backpressure',
              'ack','poll','cancel','invalid_request','conflict','cache_hit'}
SAFE_OUTCOMES = StreamGateway.terminal_states | {'active','created','reused','invalid','conflict','expired'}
METRIC_LABEL_SPECS = {
    'service_request_attempts_total': {'result': {'created','reused','expired','invalid','conflict','capacity_rejected'}},
    'service_requests_total': {'version': {'v0001','v0002'}, 'result': {'accepted','rejected'}},
    'service_terminal_total': {'version': {'v0001','v0002'}, 'reason': set(StreamGateway.terminal_states)},
    'service_output_tokens_total': {'version': {'v0001','v0002'}},
    'service_prefill_tokens_total': {'version': {'v0001','v0002'}, 'kind': {'computed','cache_hit'}},
    'service_backpressure_ticks_total': {'version': {'v0001','v0002'}},
    'service_retry_total': {'version': {'v0001','v0002'}},
}
HISTOGRAM_LABEL_SPECS = {
    'service_latency_ticks': {'version': {'v0001','v0002'}},
    'service_ttfc_ticks': {'version': {'v0001','v0002'}},
    'service_input_tokens': {'version': {'v0001','v0002'}},
    'service_output_tokens': {'version': {'v0001','v0002'}},
}


def strict_labels(name:str, labels:dict, specs:dict) -> tuple:
    require(name in specs, '未注册的指标名称')
    require(type(labels) is dict and set(labels)==set(specs[name]),
            '指标维度必须严格匹配低基数白名单')
    require(all(type(labels[k]) is str and labels[k] in allowed
                for k,allowed in specs[name].items()),'指标label值不在白名单')
    return tuple(sorted(labels.items()))


def nearest_rank(values, p):
    """教学离散样本p分位：sort，ceil(p*N)位置；N小不得当真实毫秒SLO证据。"""
    require(type(values) is list and bool(values),'分位数需要非空列表')
    require(type(p) in (float,int) and math.isfinite(p) and 0<p<=1,'分位点非法')
    require(all(type(x) in (int,float) and math.isfinite(x) and x>=0 for x in values),
            '值必须是非负、有限的逻辑tick')
    return sorted(values)[math.ceil(p*len(values))-1]


class SafeMetrics:
    def __init__(self):
        self.counters = defaultdict(int)
        self.histograms = defaultdict(list)

    def count(self,name,labels,delta=1):
        key=(name,strict_labels(name,labels,METRIC_LABEL_SPECS))
        require(type(delta) is int and delta>=0,'计数必须非负整数')
        self.counters[key]+=delta

    def observe(self,name,labels,value):
        key=(name,strict_labels(name,labels,HISTOGRAM_LABEL_SPECS))
        require(type(value) in (int,float) and math.isfinite(value) and value>=0,
                '不能记录无效、负数或物理单位不明的观测值')
        self.histograms[key].append(value)

    def count_value(self,name,labels):
        return self.counters.get((name,strict_labels(name,labels,METRIC_LABEL_SPECS)),0)

    def totals(self):
        return [{'name':name,'labels':dict(labels),'value':value}
                for (name,labels),value in sorted(self.counters.items())]

    def distributions(self):
        return [{'name':name,'labels':dict(labels),'count':len(vals),
                 'p50':nearest_rank(vals,.50),'p95':nearest_rank(vals,.95),
                 'p99':nearest_rank(vals,.99),'values':sorted(vals)}
                for (name,labels),vals in sorted(self.histograms.items()) if vals]

    def audit(self):
        require(len(self.counters)<=32 and len(self.histograms)<=8,
                'series上限越界；检查高基数标签泄漏')
        for name,labels in self.counters:
            strict_labels(name,dict(labels),METRIC_LABEL_SPECS)
        for name,labels in self.histograms:
            strict_labels(name,dict(labels),HISTOGRAM_LABEL_SPECS)
        require(all(v>=0 for v in self.counters.values()),'计数器不能递减')
        return True


class SafeTraceLog:
    """自定义教学格式，不导出标准OTLP；只有服务生成的关联ID，无用户文字与token。"""
    def __init__(self,seed=7801):
        self.seed=seed
        self.roots={}
        self.spans=[]
        self.logs=[]
        self.sequence=0

    def _ids(self,sid):
        # 仅为教学的可复现伪随机ID，不应用于真实生产token/安全边界。
        trace=digest({'lesson':VERSION,'seed':self.seed,'request_sequence':sid})[:32]
        require(TRACE_ID_RE.fullmatch(trace) is not None,'trace_id格式错误')
        return trace

    def _span_id(self,trace):
        self.sequence+=1
        return digest({'trace':trace,'number':self.sequence})[:16]

    def _attributes(self,attrs):
        require(type(attrs) is dict and set(attrs)<=SPAN_ATTRIBUTES,'span属性含高风险/未知字段')
        safe={}
        for k,v in attrs.items():
            if k in ('model_version','version'):
                require(v in {'v0001','v0002'},'未授权的model版本')
            elif k in ('terminal_reason','response_status'):
                require(v in SAFE_OUTCOMES,'未知终止状态')
            elif k in ('model_fingerprint','template_fingerprint'):
                require(type(v) is str and re.fullmatch(r'[0-9a-f]{64}',v) is not None,
                        '模型/模板身份必须完整哈希，不允许用户文字')
            else:
                require(type(v) is int and 0<=v<=10000000, 'span仅允许有界整数属性')
            safe[k]=v
        return safe

    def open(self,sid,tick,version,model_fingerprint,template_fingerprint):
        require(sid not in self.roots and type(tick) is int and tick>=0,'重复或非法root')
        trace=self._ids(sid)
        require(trace not in (r['trace_id'] for r in self.roots.values()),'trace ID冲突')
        root={'trace_id':trace,'span_id':self._span_id(trace),'parent_span_id':None,
              'name':'inference.request','start_tick':tick,'end_tick':None,
              'status':'running','attributes':self._attributes({
                  'model_version':version,'model_fingerprint':model_fingerprint,
                  'template_fingerprint':template_fingerprint})}
        self.roots[sid]=root;self.spans.append(root)
        return root

    def add_span(self,sid,name,start,end,attrs=None,status='ok'):
        require(sid in self.roots and name in SPAN_NAMES and name!='inference.request','span根或名称非法')
        require(type(start) is int and type(end) is int and
                self.roots[sid]['start_tick']<=start<=end,'span开始结束tick非法')
        parent=self.roots[sid]
        require(parent['end_tick'] is None or end<=parent['end_tick'],
                '子span不能超出已关闭的root')
        require(status in {'ok','paused','error','rejected'},'span状态非法')
        data={'trace_id':parent['trace_id'],'span_id':self._span_id(parent['trace_id']),
              'parent_span_id':parent['span_id'],'name':name,'start_tick':start,
              'end_tick':end,'status':status,'attributes':self._attributes(attrs or {})}
        self.spans.append(data)
        return data

    def log(self,sid,event,tick,span=None,attrs=None):
        require(sid in self.roots and event in LOG_EVENTS,'日志只能使用白名单事件')
        require(type(tick) is int and tick>=self.roots[sid]['start_tick'],'日志tick非法')
        attrs=self._attributes(attrs or {})
        root=self.roots[sid]
        span_id=root['span_id'] if span is None else span['span_id']
        require(any(s['span_id']==span_id and s['trace_id']==root['trace_id']
                    for s in self.spans),'日志关联到未知span')
        self.logs.append({'trace_id':root['trace_id'],'span_id':span_id,
                          'event':event,'tick':tick,'attributes':attrs})

    def finish(self,sid,tick,status):
        require(sid in self.roots and status in StreamGateway.terminal_states,'终态错误')
        root=self.roots[sid]
        require(root['end_tick'] is None and type(tick) is int and tick>=root['start_tick'],
                '同一请求只能关闭一次')
        root['end_tick']=tick;root['status']=status
        root['attributes']['terminal_reason']=status

    def audit(self,active_sids=None):
        active_sids=set(active_sids or ())
        spans_by_id={s['span_id']:s for s in self.spans}
        require(len(spans_by_id)==len(self.spans),'span ID不唯一')
        all_traces={r['trace_id'] for r in self.roots.values()}
        require(len(all_traces)==len(self.roots),'trace ID不唯一')
        require(len(self.spans)>=len(self.roots),'至少每个请求有root')
        for sid,root in self.roots.items():
            require(root['parent_span_id'] is None and root['name']=='inference.request',
                    'root不存在或错误')
            require((root['end_tick'] is None)==(sid in active_sids),
                    'trace终态与实际会话不一致')
        for span in self.spans:
            require(TRACE_ID_RE.fullmatch(span['trace_id']) is not None and
                    SPAN_ID_RE.fullmatch(span['span_id']) is not None,'TraceID/SpanID格式不符')
            require(span['name'] in SPAN_NAMES and type(span['start_tick']) is int and
                    (span['end_tick'] is None or span['end_tick']>=span['start_tick']),
                    'span名称或时间非法')
            self._attributes(span['attributes'])
            if span['parent_span_id'] is not None:
                require(span['parent_span_id'] in spans_by_id,'悬空parent span')
                parent=spans_by_id[span['parent_span_id']]
                require(parent['trace_id']==span['trace_id'],'跨请求错误关联')
                require(parent['start_tick']<=span['start_tick'] and
                        (parent['end_tick'] is None or span['end_tick']<=parent['end_tick']),
                        '子span跨越root时间边界')
        for line in self.logs:
            require(line['trace_id'] in all_traces and line['span_id'] in spans_by_id,
                    '日志没有合法Trace关联')
            self._attributes(line['attributes'])
            require(line['event'] in LOG_EVENTS,'非法日志事件')
        return {'trace_roots':len(self.roots),'span_count':len(self.spans),'log_count':len(self.logs)}


class ServingTelemetry:
    def __init__(self,seed=7801):
        self.trace=SafeTraceLog(seed)
        self.metrics=SafeMetrics()
        self.first_chunk={}
        self.finished={}
        self.attempts=0

    def count_attempt(self,result):
        self.attempts+=1
        self.metrics.count('service_request_attempts_total',{'result':result})

    def admitted(self,record,result,tick,model_fingerprint):
        sid=record.sid;version=record.request.version
        self.trace.open(sid,tick,version,model_fingerprint,digest(record.request.template_revision))
        if result['status']=='created':
            attrs={'model_version':version,'new_compute':1}
            self.trace.add_span(sid,'request.admission',tick,tick,attrs)
            pre=record.prefill
            require(pre is not None,'已接受请求缺少真实prefill统计')
            self.trace.add_span(sid,'inference.prefill',tick,tick,
                                {'model_version':version,'input_tokens':pre['prompt_tokens'],
                                 'cache_hit_tokens':pre['hit_tokens'],
                                 'new_prefill_tokens':pre['new_tokens_computed']})
            self.metrics.count('service_requests_total',{'version':version,'result':'accepted'})
            self.metrics.count('service_prefill_tokens_total',{'version':version,'kind':'computed'},
                               pre['new_tokens_computed'])
            self.metrics.count('service_prefill_tokens_total',{'version':version,'kind':'cache_hit'},
                               pre['hit_tokens'])
            self.metrics.observe('service_input_tokens',{'version':version},pre['prompt_tokens'])
            self.trace.log(sid,'admitted',tick,attrs={'model_version':version,
                                                     'input_tokens':pre['prompt_tokens']})
            if pre['hit_tokens']:
                self.trace.log(sid,'cache_hit',tick,attrs={'model_version':version,
                    'cache_hit_tokens':pre['hit_tokens']})
        elif result['status']=='capacity_rejected':
            self.trace.add_span(sid,'request.admission',tick,tick,
                {'model_version':version,'response_status':'capacity_rejected'},'rejected')
            self.metrics.count('service_requests_total',{'version':version,'result':'rejected'})
            self.trace.log(sid,'rejected',tick,attrs={'model_version':version,
                                                     'response_status':'capacity_rejected'})
            self.finish(record,tick)
        else:raise LessonError('不是新建操作，不应增加trace')

    def on_event(self,record,event,tick):
        sid=record.sid
        if sid not in self.trace.roots:return
        kind=event['type']
        if kind=='delta':
            self.trace.log(sid,'token_committed',tick,attrs={'generated_tokens':len(record.output_ids)})
            if sid not in self.first_chunk:
                self.first_chunk[sid]=tick
                self.metrics.observe('service_ttfc_ticks',{'version':record.request.version},
                                     tick-record.accepted_tick)

    def finish(self,record,tick):
        if record.sid not in self.trace.roots:return
        require(record.sid not in self.finished,'重复终态不能重复计数')
        require(record.state in StreamGateway.terminal_states,'只有终态才记完成')
        version=self.trace.roots[record.sid]['attributes']['model_version']
        self.trace.finish(record.sid,tick,record.state)
        self.trace.log(record.sid,'terminal',tick,
                       attrs={'model_version':version,'terminal_reason':record.state,
                              'generated_tokens':len(record.output_ids),'kv_live_after':0})
        self.metrics.count('service_terminal_total',{'version':version,'reason':record.state})
        self.metrics.observe('service_latency_ticks',{'version':version},tick-record.accepted_tick)
        self.metrics.observe('service_output_tokens',{'version':version},len(record.output_ids))
        self.finished[record.sid]={'state':record.state,'end_tick':tick,
                                   'start_tick':record.accepted_tick,'version':version,
                                   'first_chunk_tick':self.first_chunk.get(record.sid)}

    def verify(self,service):
        active={r.sid for r in service.records.values() if r.state=='active'}
        self.trace.audit(active)
        self.metrics.audit()
        require(len(self.trace.roots)==len(service.records),'逻辑请求与root不守恒')
        require(len(self.finished)==len(service.records)-len(active),'完成请求与终态计数不守恒')
        require(sum(v for (name,_),v in self.metrics.counters.items()
                    if name=='service_terminal_total')==len(self.finished),'终态指标总数不守恒')
        require(sum(v for (name,_),v in self.metrics.counters.items()
                    if name=='service_output_tokens_total')==service.total_tokens,
                '生成计数器与真实KV服务已生成token数不一致')
        prefill_computed=sum(v for (name,labels),v in self.metrics.counters.items()
                             if name=='service_prefill_tokens_total' and dict(labels)['kind']=='computed')
        prefill_hit=sum(v for (name,labels),v in self.metrics.counters.items()
                        if name=='service_prefill_tokens_total' and dict(labels)['kind']=='cache_hit')
        require(prefill_computed+service.total_tokens==service.runner.runner.tokens_processed,
                'prefill＋decode token指标与实际模型调用不守恒')
        require(prefill_hit==service.pool.hit_tokens,
                '缓存命中统计与实际页池不守恒')
        require(sum(v for (name,_),v in self.metrics.counters.items()
                    if name=='service_request_attempts_total')==self.attempts,'重试/首次attempt计数不守恒')
        for r in service.records.values():
            if r.state!='active':
                require(r.sid not in service.pool.sessions,'终态没有释放KV session')
                require(self.finished[r.sid]['state']==r.state,'终态与trace不一致')
        return {'attempts':self.attempts,'logical_requests':len(self.trace.roots),
                'active':len(active),'terminal':len(self.finished),
                'generated_tokens':service.total_tokens,
                'ttfc_observations':len(self.first_chunk)}

    def report(self,service):
        reconciliation=self.verify(service)
        return {'reconciliation':reconciliation,
                'spans':copy.deepcopy(self.trace.spans),
                'structured_logs':copy.deepcopy(self.trace.logs),
                'metric_counters':self.metrics.totals(),
                'metric_histograms':self.metrics.distributions(),
                'scope':'single-process logical ticks; custom schema, not OTLP'}

In [15]:
tr=SafeTraceLog(seed=7801)
root=tr.open('rs0000',0,'v0001',digest('model_example'),digest('template_example'))
child=tr.add_span('rs0000','inference.prefill',0,0,{'input_tokens':8})
tr.log('rs0000','admitted',0,child,{'input_tokens':8})
tr.finish('rs0000',2,'done')
print('root:',root)
print('child parent matches root:',child['parent_span_id']==root['span_id'])
print('trace audit:',tr.audit())

root: {'trace_id': '78f75f27896c4e9be7669b55b37823e7', 'span_id': '8d04128539330779', 'parent_span_id': None, 'name': 'inference.request', 'start_tick': 0, 'end_tick': 2, 'status': 'done', 'attributes': {'model_version': 'v0001', 'model_fingerprint': '7aa3f32111e2bf40f776503f78791646d1113128aa27674e33b7146e10ee3465', 'template_fingerprint': '45161e2fc0c0f9744ac8ddf00e995725321005f27e99ee3abb08864ee920e680', 'terminal_reason': 'done'}}
child parent matches root: True
trace audit: {'trace_roots': 1, 'span_count': 2, 'log_count': 1}


### 请求已经结束，重试又来了：还要追加span吗？

不要在已结束的 root 时间范围之外硬加新子span。本课对结束后的幂等重试记录**与原 trace 关联的日志**，但不新建一个越过根 span 终点的子 span；活跃期间的重试则可以记录子 span。这个简化演示的是本地关联规则，不是完整跨服务重试 trace 设计。

In [16]:
late=SafeTraceLog(seed=7802)
late.open('rs0001',0,'v0001',digest('model'),digest('template'))
late.finish('rs0001',1,'done')
try:
    late.add_span('rs0001','request.retry',3,3)
except LessonError as error:
    print('预期拒绝：',error)
late.log('rs0001','retry',3,attrs={'new_compute':0})
print('晚到重试日志仍可关联：',late.logs[-1]['trace_id']==late.roots['rs0001']['trace_id'])

预期拒绝： 子span不能超出已关闭的root
晚到重试日志仍可关联： True


## 5. 结构化日志：先定义允许什么，而不是事后删除敏感文本

日志允许：白名单事件（`admitted/token_committed/backpressure/terminal/...`）、`trace_id`、`span_id`、逻辑 tick，以及受限的计数和终态属性。

日志**不允许**：`prompt`、`text`、`principal`、`idempotency_key`、`token_id`、不受限的用户标识或错误堆栈原文。用户提供的文本可能含密钥；默认不把请求、响应、tokens 写入日志。

同一模型版本可从root上的hash核对权重，日志中的关联ID不是授权凭证。纯哈希或遮盖字符串也不自动保证不可反推，应结合访问控制、密钥管理和日志保留期限。当前可复现教学ID不是生产环境的安全令牌。

In [17]:
privacy=SafeTraceLog(seed=7803)
privacy.open('rs0002',0,'v0002',digest('safe_model'),digest('safe_template'))
for attr in ({'prompt':'PrivateSecret'}, {'token_id':22}, {'user_email':'secret@example.com'}):
    try:
        privacy.log('rs0002','admitted',0,attrs=attr)
    except LessonError as error:
        print('拒绝原始/非白名单属性：',list(attr), '→', str(error))

拒绝原始/非白名单属性： ['prompt'] → span属性含高风险/未知字段
拒绝原始/非白名单属性： ['token_id'] → span属性含高风险/未知字段
拒绝原始/非白名单属性： ['user_email'] → span属性含高风险/未知字段


## 6. Metrics：Counter、Histogram 与高基数陷阱

Trace 与日志面向单条操作；指标用于聚合。Prometheus 的每一个不同标签组合都是一个新时间序列，不能把请求ID、用户邮箱、任意用户文本作为 label。[Prometheus 标签实践](https://prometheus.io/docs/practices/naming/)

本课只允许：模型版本（两种）、终态类别（有限集合）、请求/重试结果和缓存统计类型。任何新增字段或未知值都拒绝写入。

Counter：请求提交尝试、逻辑请求、终态、真正生成的 token、真实 prefill 计算量、cache hit、背压 tick。

Histogram：服务完成逻辑 tick、首次实际 delta 的逻辑 tick、输入/输出 token 数。用离散 nearest-rank p50/p95/p99 做**小样本教学**，不把它当 Prometheus histogram_quantile 的插值输出，也不称为 GPU 延迟。

首次 chunk 的分母只包含**实际发布过 delta 的请求**。EOS 首token或直接拒绝没有这项观测，不能填0拉低延迟。

In [18]:
m=SafeMetrics()
m.count('service_requests_total',{'version':'v0001','result':'accepted'})
m.observe('service_latency_ticks',{'version':'v0001'},3)
m.observe('service_latency_ticks',{'version':'v0001'},10)
print('Counters:',m.totals())
print('Histogram:',m.distributions())
try:
    m.count('service_requests_total',{'version':'v0001','result':'accepted','request_id':'rs123'})
except LessonError as error:
    print('高基数标签被拒绝：',error)

Counters: [{'name': 'service_requests_total', 'labels': {'result': 'accepted', 'version': 'v0001'}, 'value': 1}]
Histogram: [{'name': 'service_latency_ticks', 'labels': {'version': 'v0001'}, 'count': 2, 'p50': 3, 'p95': 10, 'p99': 10, 'values': [3, 10]}]
高基数标签被拒绝： 指标维度必须严格匹配低基数白名单


### 哪个分位数公式？

本课离散近邻秩（nearest rank）：对 N 个样本排序，取第 `ceil(p*N)` 个。5个点 `[1,2,3,4,8]` 的 p95 是 **8**，不是假装大样本下有精密的尾延迟稳定估计。小N的p95/p99通常就是最大值，应同时报告样本数量。

真实服务监控应使用适当bucket/原始直方图聚合，避免把不同请求类别的p95直接平均。OpenTelemetry GenAI 语义规范也区分累计 token 使用量和按操作的 token 数分布；此处为本地教学字段名，并未声称采用其现行完整规范。[OTel GenAI规范](https://opentelemetry.io/docs/specs/semconv/gen-ai/)

In [19]:
samples=[8,2,3,1,4]
for p in (.25,.5,.95,.99):
    print(f'p{int(p*100):02d}: {nearest_rank(samples,p)} ticks (N={len(samples)})')

p25: 2 ticks (N=5)
p50: 3 ticks (N=5)
p95: 8 ticks (N=5)
p99: 8 ticks (N=5)


## 7. SLI 的分母必须写清楚

这节只度量**服务完成**，不是回答正确率。教学 SLI 规定：

- `eligible`：所有已建立逻辑操作，排除客户端主动取消。
- `good`：终态是 `done`，且从接收至终态不超过 **8 tick**。
- `bad`：容量拒绝、慢消费者、截止时间失败、KV OOM，以及完成过慢。
- **重复提交attempt不重复成为新操作**。

`SLI=good/eligible`，eligible为0时返回 `None`，不能写0%或100%。取消是否排除是**这份明确制定的服务契约**，不同业务必须重新商定。

`offered`、`attempts`、`eligible`、`done` 和 TTFC 观测数有不同分母。下面先算一组可手算样本。

In [20]:
# %% 13 观测式服务包装器：不修改上一课真实KV与流式契约
class InstrumentedGateway(StreamGateway):
    """覆盖公开动作和成功KV提交，Trace是观测副作用而非替代计算。"""
    def __init__(self,tok,models,authority,config=StreamConfig(),seed=7801):
        super().__init__(tok,models,authority,config)
        self.telemetry=ServingTelemetry(seed)

    def _emit(self,record,kind,data):
        result=super()._emit(record,kind,data)
        self.telemetry.on_event(record,result,self.now)
        return result

    def _terminate(self,record,kind,data):
        super()._terminate(record,kind,data)
        if record.sid in self.telemetry.trace.roots:
            require(record.sid not in self.pool.sessions,'终态释放KV后才能记录terminal')
            self.telemetry.trace.add_span(record.sid,'kv.release',self.now,self.now,
                    {'kv_live_after':0,'generated_tokens':len(record.output_ids)})
            self.telemetry.finish(record,self.now)

    def submit(self,request):
        try:
            response=super().submit(request)
        except LessonError:
            self.telemetry.count_attempt('invalid')
            raise
        kind=response['status']
        self.telemetry.count_attempt(kind)
        record=self.sid_index[response['sid']]
        if kind in {'created','capacity_rejected'}:
            self.telemetry.admitted(record,response,self.now,
                    self.runner.runner.model_hashes[record.request.version])
            # accepted/terminal事件已在super().submit里发布；不重复计算/发布。
        else:
            require(kind in {'reused','expired'},'重复提交返回未知状态')
            version=self.telemetry.trace.roots[record.sid]['attributes']['model_version']
            self.telemetry.metrics.count('service_retry_total',{'version':version})
            root=self.telemetry.trace.roots[record.sid]
            span=(self.telemetry.trace.add_span(record.sid,'request.retry',self.now,self.now,
                        {'model_version':version,'new_compute':0})
                  if root['end_tick'] is None or self.now<=root['end_tick'] else None)
            # 请求结束后的重试只是相关日志，不能把新span塞进已关闭的root时间区间。
            self.telemetry.trace.log(record.sid,'retry',self.now,span,{'model_version':version,'new_compute':0})
        return response

    def _generate_one(self,record):
        previous=len(record.output_ids)
        super()._generate_one(record)
        if len(record.output_ids)>previous:
            version=self.telemetry.trace.roots[record.sid]['attributes']['model_version']
            self.telemetry.trace.add_span(record.sid,'inference.decode',self.now-1,self.now,
                    {'model_version':version,'generated_tokens':1})
            self.telemetry.metrics.count('service_output_tokens_total',{'version':version})

    def tick(self,steps=1):
        require(type(steps) is int and 1<=steps<=1000,'tick步数非法')
        for _ in range(steps):
            old_tick=self.now
            paused=[r for r in self._active()
                    if len(r.event_log)-1-r.acked_seq>=self.config.max_unacked
                    and old_tick+1<r.deadline_tick]
            skips=self.backpressure_skips
            super().tick(1)
            if self.backpressure_skips>skips:
                require(self.backpressure_skips-skips==len(paused),
                        '背压计数与trace候选不守恒')
                for record in paused:
                    version=self.telemetry.trace.roots[record.sid]['attributes']['model_version']
                    pending=len(record.event_log)-1-record.acked_seq
                    span=self.telemetry.trace.add_span(record.sid,'stream.backpressure',self.now,self.now,
                        {'model_version':version,'pending_events':pending},status='paused')
                    self.telemetry.trace.log(record.sid,'backpressure',self.now,span,
                                            {'model_version':version,'pending_events':pending})
                    self.telemetry.metrics.count('service_backpressure_ticks_total',{'version':version})

    def ack(self,principal,key,last_event_id):
        result=super().ack(principal,key,last_event_id)
        record=self._record(principal,key)
        root=self.telemetry.trace.roots[record.sid]
        span=(self.telemetry.trace.add_span(record.sid,'stream.ack',self.now,self.now,
                              {'pending_events':result['pending']})
              if root['end_tick'] is None or self.now<=root['end_tick'] else None)
        self.telemetry.trace.log(record.sid,'ack',self.now,span,
                                 {'pending_events':result['pending']})
        return result

    def poll(self,principal,key,last_event_id=None):
        records=super().poll(principal,key,last_event_id)
        record=self._record(principal,key)
        root=self.telemetry.trace.roots[record.sid]
        span=(self.telemetry.trace.add_span(record.sid,'stream.poll',self.now,self.now,
                              {'event_count':len(records)})
              if root['end_tick'] is None or self.now<=root['end_tick'] else None)
        self.telemetry.trace.log(record.sid,'poll',self.now,span,{'event_count':len(records)})
        return records

    def cancel(self,principal,key):
        result=super().cancel(principal,key)
        record=self._record(principal,key)
        root=self.telemetry.trace.roots[record.sid]
        span=(self.telemetry.trace.add_span(record.sid,'stream.cancel',self.now,self.now,
                    {'response_status':record.state})
              if root['end_tick'] is None or self.now<=root['end_tick'] else None)
        self.telemetry.trace.log(record.sid,'cancel',self.now,span,
                    {'response_status':record.state})
        return result

    def audit_instrumentation(self):
        base=self.audit()
        return {'gateway':base,'telemetry':self.telemetry.verify(self)}


@dataclass(frozen=True)
class SLOConfig:
    success_latency_ticks: int = 8
    target: float = 0.90
    short_window_ticks: int = 5
    long_window_ticks: int = 14
    min_short: int = 3
    min_long: int = 7
    burn_threshold: float = 2.0

    def validate(self):
        for key in ('success_latency_ticks','short_window_ticks','long_window_ticks',
                    'min_short','min_long'):
            require(type(getattr(self,key)) is int and getattr(self,key)>0,key+'必须是正整数')
        require(self.short_window_ticks<self.long_window_ticks,'短窗口必须小于长窗口')
        require(type(self.target) in (int,float) and math.isfinite(self.target)
                and 0<self.target<1,'target非法')
        require(type(self.burn_threshold) in (int,float) and math.isfinite(self.burn_threshold)
                and self.burn_threshold>0,'burn阈值非法')


def terminal_sli(records:list[dict],cfg:SLOConfig)->dict:
    """提供服务完成SLI，不测量答案正确性；client取消排除，其余终态计入。"""
    cfg.validate()
    require(type(records) is list,'终态数据需要列表')
    rows=[]
    for r in records:
        require(type(r) is dict and set(r)=={'accepted_tick','end_tick','state'},
                '不能携带用户负载、ID到SLI入口')
        start,end,state=r['accepted_tick'],r['end_tick'],r['state']
        require(type(start) is int and type(end) is int and 0<=start<=end,'终态时间非法')
        require(state in StreamGateway.terminal_states,'SLI输入只能是终态')
        eligible=state!='cancelled'
        good=eligible and state=='done' and end-start<=cfg.success_latency_ticks
        rows.append({'end_tick':end,'eligible':eligible,'good':good,'bad':eligible and not good})
    eligible=sum(x['eligible'] for x in rows);good=sum(x['good'] for x in rows)
    observed=[r['end_tick']-r['accepted_tick'] for r in records if r['state']=='done']
    return {'offered':len(rows),'eligible':eligible,'excluded_client_cancellations':len(rows)-eligible,
            'good':good,'bad':eligible-good,'good_fraction':good/eligible if eligible else None,
            'p95_done_latency_ticks':nearest_rank(observed,.95) if observed else None,
            'definition':'terminal done within logical tick budget; rejections+timeout count bad; cancellation excluded',
            'events':rows}


def dual_window_alert(events:list[dict],now:int,cfg:SLOConfig)->dict:
    """教学用双窗口burn率门控；不是照搬真实99.9%告警秒/分钟配置。"""
    cfg.validate()
    require(type(now) is int and now>=0,'now必须非负tick')
    def window(width,min_count):
        selected=[r for r in events if r['eligible'] and now-width<r['end_tick']<=now]
        count=len(selected);bad=sum(x['bad'] for x in selected)
        ratio=bad/count if count else None
        burn=ratio/(1-cfg.target) if ratio is not None else None
        return {'window_ticks':width,'eligible':count,'bad':bad,'bad_fraction':ratio,
                'burn_rate':burn,'sufficient_volume':count>=min_count,
                'breached':count>=min_count and burn is not None and burn>=cfg.burn_threshold}
    require(type(events) is list and all(type(e) is dict and
            set(e)=={'end_tick','eligible','good','bad'} and
            type(e['end_tick']) is int and e['end_tick']>=0 and
            type(e['eligible']) is bool and type(e['bad']) is bool and
            type(e['good']) is bool and
            not(e['good'] and e['bad']) and (e['good'] or e['bad'] or not e['eligible'])
            for e in events),'告警输入类型/状态错误')
    short=window(cfg.short_window_ticks,cfg.min_short)
    long=window(cfg.long_window_ticks,cfg.min_long)
    return {'alert':short['breached'] and long['breached'],
            'short':short,'long':long,
            'note':'logical-tick teaching window; count guard; not real burn-rate paging policy'}


def diagnose(report:dict)->list[dict]:
    """仅输出有计数证据的相关信号，不推断真实根因。"""
    counts={x['name']+':'+canonical(x['labels']):x['value'] for x in report['metric_counters']}
    def subtotal(metric,pred=lambda _:True):
        return sum(x['value'] for x in report['metric_counters']
                   if x['name']==metric and pred(x['labels']))
    signals=[]
    def add(code,evidence,meaning):
        signals.append({'signal':code,'evidence_count':evidence,'interpretation':meaning,
                        'certainty':'correlated_signal_not_proven_root_cause'})
    backpressure=subtotal('service_backpressure_ticks_total')
    deadline=subtotal('service_terminal_total',lambda x:x['reason']=='deadline_exceeded')
    saturation=subtotal('service_terminal_total',lambda x:x['reason'] in {'kv_oom','capacity_rejected'})
    slow=subtotal('service_terminal_total',lambda x:x['reason']=='slow_consumer')
    if backpressure:add('client_backpressure',backpressure,'客户端确认窗口饱和；检查ack和消费速率')
    if slow:add('slow_consumer_terminal',slow,'服务按租约关闭慢消费者；不等于模型推理错误')
    if deadline:add('deadline_exceeded',deadline,'绝对截止时间触发；检查排队/慢消费/生成预算')
    if saturation:add('admission_or_kv_capacity',saturation,'容量拒绝或KV不足；核对页池和并发')
    if not signals:add('no_known_fault_signal',0,'当前计数中未发现已建模的失败类型，不代表服务无故障')
    return signals


def alert_examples()->dict:
    cfg=SLOConfig()
    def make(i,good,eligible=True):return {'end_tick':i,'eligible':eligible,'good':good and eligible,
                                          'bad':not good and eligible}
    # 14个终态落在长窗口，最近5个均失败。
    incident=[make(i,i<8) for i in range(1,15)]
    quiet=[make(i,True) for i in range(1,15)]
    low_volume=[make(12,False),make(13,False)]
    return {'sustained':dual_window_alert(incident,14,cfg),
            'healthy':dual_window_alert(quiet,14,cfg),
            'low_volume':dual_window_alert(low_volume,14,cfg)}

In [21]:
slo_cfg=SLOConfig(success_latency_ticks=8)
small_records=[
 {'accepted_tick':0,'end_tick':8,'state':'done'},
 {'accepted_tick':0,'end_tick':9,'state':'done'},
 {'accepted_tick':1,'end_tick':1,'state':'capacity_rejected'},
 {'accepted_tick':1,'end_tick':4,'state':'cancelled'},
]
small_sli=terminal_sli(small_records,slo_cfg)
print({k:v for k,v in small_sli.items() if k!='events'})

{'offered': 4, 'eligible': 3, 'excluded_client_cancellations': 1, 'good': 1, 'bad': 2, 'good_fraction': 0.3333333333333333, 'p95_done_latency_ticks': 9, 'definition': 'terminal done within logical tick budget; rejections+timeout count bad; cancellation excluded'}


## 8. 双窗口告警：坏率、错误预算与最小样本数

教学规则目标 `SLO=90%`，所以预算是10%错误；若某窗口坏率50%，则其 **burn rate=50%/10%=5**。

只有同时满足：

- 短窗5tick内至少3个 eligible，烧损率大于等于2；
- 长窗14tick内至少7个 eligible，烧损率也大于等于2；

才触发教学告警。另设“健康窗口”和“低流量时虽然都错也不立即告警”的用例。

这只是演示多窗口思想。生产SLO告警通常按真实时间、错误预算、业务流量与告警成本设计，**不能复制这些 tick 和90%阈值直接上线**。[Google SRE：Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/)

In [22]:
for name,val in alert_examples().items():
    print(name, {'alert':val['alert'],
                 'short':(val['short']['eligible'],val['short']['bad'],val['short']['burn_rate']),
                 'long':(val['long']['eligible'],val['long']['bad'],val['long']['burn_rate'])})

sustained {'alert': True, 'short': (5, 5, 10.000000000000002), 'long': (14, 7, 5.000000000000001)}
healthy {'alert': False, 'short': (5, 0, 0.0), 'long': (14, 0, 0.0)}
low_volume {'alert': False, 'short': (2, 2, 10.000000000000002), 'long': (2, 2, 10.000000000000002)}


## 9. 观测不能证明根因：信号先于结论

根据终态与背压 Counter，我们会给出：慢消费者确认窗口、绝对超时、准入/KV容量的**相关线索**，并标注 `correlated_signal_not_proven_root_cause`。

例如 `deadline_exceeded` 并不证明“模型计算太慢”：它也可能因为队列、消费端、截止配置或系统资源造成。没有异常不代表没有故障；没有原始用户内容时，必须通过受控的业务排查渠道定位质量问题。

使用方法：先按低基数指标发现异常 → 选择特定终态 → 按 trace_id 钻取 root、prefill、decode、release 和受限事件 → 对照资源审计，而不是把所有日志无差别全文搜索。

## 10. 完整真实 KV 演练

下面同一进程中实际执行：正常完成、同键重试、SSE重复投递去重、共享前缀缓存命中、另一租户/LoRA版本、绝对超时、慢消费者、容量拒绝、显式取消、TTL禁止重新计算与最终 KV 页清零。

带有 `PrivatePaymentTokenNeverLog123` 的测试提示将确实进入模型的输入，但**遥测产物必须不出现该文本或原始token ID字段**。这是负向隐私测试，而不是对任意第三方日志或系统内存的全面保密认证。

注意第78课所有延迟都只用逻辑 tick；真实CPU张量计算不等于对外提供了可接受的吞吐或响应时间。

In [23]:
# %% 14 真实KV场景：正确路径、重复请求、两种版本、背压、过载、超时、取消与TTL

def new_monitored_service(seed=7801,config=None,model_bundle=None):
    torch.set_num_threads(1)
    tok,models=build_models(seed) if model_bundle is None else model_bundle
    authority=TrustedScopeAuthority({'alice':'tenant-A','bob':'tenant-B','alice2':'tenant-A'})
    return InstrumentedGateway(tok,models,authority,config or StreamConfig(),seed=seed)


def real_service_scenarios(seed=7801,model_bundle=None):
    cfg=StreamConfig(max_sessions=3,max_unacked=2,slow_consumer_ticks=3,
                     retention_ticks=5,capacity_blocks=96,cache_capacity_blocks=12)
    svc=new_monitored_service(seed,cfg,model_bundle)
    scenarios={}
    secret='PrivatePaymentTokenNeverLog123'
    request=safe_request('same','hello',timeout=30,max_tokens=4)
    a=svc.submit(request);retry=svc.submit(request)
    require(a['status']=='created' and retry['status']=='reused' and a['sid']==retry['sid'],
            '重复提交未复用计算')
    client=ReplayClient('alice','same');client.receive(svc)
    for _ in range(5):
        if svc._record('alice','same').state!='active':break
        svc.tick(1);client.receive(svc)
    a_record=svc._record('alice','same')
    require(a_record.state=='done','正常请求未完成')
    client.restart_without_cursor();client.receive(svc)
    require(client.duplicates>=1,'测试应覆盖按ID去重')
    # 同租户共享前缀：两次计算都是真实QKV张量，是否命中从runner统计记录。
    second=svc.submit(safe_request('next','hello',principal='alice2',version='v0001',
                                   timeout=30,max_tokens=4))
    second_record=svc.sid_index[second['sid']]
    cache_hit=second_record.prefill['hit_tokens']
    svc.cancel('alice2','next')
    # 不同租户和不同模型版本，必须有自己的trace/服务身份。
    other=svc.submit(safe_request('other',secret,principal='bob',version='v0002',timeout=1,max_tokens=3))
    svc.tick(1)
    require(svc.sid_index[other['sid']].state=='deadline_exceeded','超时路径未命中')
    # 消费者不确认：窗口达到上限，发生暂停与慢消费者关闭。
    paused=svc.submit(safe_request('paused','world',timeout=20,max_tokens=9))
    svc.tick(6)
    require(svc.sid_index[paused['sid']].state=='slow_consumer','慢消费者未关闭')
    # 准入硬上限：同时打开3个会话，第四个成为合法的capacity_rejected终态。
    capacity=[]
    for index in range(4):
        response=svc.submit(safe_request(f'cap{index}','answer',timeout=30,max_tokens=8))
        capacity.append(response)
    require([x['status'] for x in capacity].count('capacity_rejected')==1,'未触发容量拒绝')
    for index in range(3):svc.cancel('alice',f'cap{index}')
    # 取消终态以及无指纹混淆的重复重试。
    cancelled=svc.submit(safe_request('cancel','42',timeout=20,max_tokens=7))
    first=svc.cancel('alice','cancel');again=svc.cancel('alice','cancel')
    require(first['cancelled_now'] and not again['cancelled_now'],'取消没有幂等性')
    repeat=svc.submit(safe_request('cancel','42',timeout=20,max_tokens=7))
    require(repeat['sid']==cancelled['sid'] and not repeat['new_compute'],'取消后不能重计算')
    # 不同文本/版本改变幂等语义的提交必须明确拒绝，不记录文字。
    try:
        svc.submit(safe_request('same','world',timeout=30,max_tokens=4))
    except LessonError:conflict_rejected=True
    else:conflict_rejected=False
    require(conflict_rejected,'冲突提交错误接受')
    scenarios['idempotency']={'same_trace_for_retries':svc.telemetry.trace.roots[a['sid']]['trace_id']==
                               svc.telemetry.trace.roots[retry['sid']]['trace_id'],
                              'duplicate_deliveries_deduplicated':client.duplicates,
                              'conflict_rejected':conflict_rejected}
    scenarios['prefill']={'shared_prefix_hit_tokens':cache_hit,
                          'real_tokens_processed':svc.runner.runner.tokens_processed,
                          'unique_models':len({x['attributes']['model_version']
                                 for x in svc.telemetry.trace.roots.values()})}
    scenarios['terminal']={'states':{r.state:sum(s.state==r.state for s in svc.records.values())
                                        for r in svc.records.values()},
                           'backpressure_ticks':svc.backpressure_skips,
                           'generated_tokens':svc.total_tokens}
    # 所有会话结束后到达TTL，只清除业务重放记录，不静默重算请求。
    require(not svc._active(),'场景结束仍有活动请求')
    svc.tick(cfg.retention_ticks)
    expired=svc.submit(request)
    require(expired['status']=='expired','TTL已过的重试不应自动重算')
    try:svc.poll('alice','same')
    except ReplayGone:gone=True
    else:gone=False
    require(gone,'TTL已经到期应拒绝重放')
    scenarios['ttl']={'expired_no_recompute':expired['status']=='expired',
                      'replay_gone':gone,'tombstones':len(svc.records)}
    physical=svc.final_flush()
    require(physical['live_blocks']==0 and physical['session_count']==0
            and not bool(svc.pool.data.count_nonzero()),'退出时物理KV未清空')
    telemetry=svc.telemetry.report(svc)
    raw=canonical(telemetry)
    require(secret not in raw and 'PrivatePayment' not in raw,'遥测泄漏原始用户文本')
    require(not any(k in raw for k in ('"idempotency_key"','"principal"','"token_id"',
                                       '"prompt"','"text"')),'遥测不应包含会话原始负载字段')
    finished=[{'accepted_tick':r.accepted_tick,'end_tick':r.end_tick,'state':r.state}
              for r in svc.records.values()]
    slo_cfg=SLOConfig()
    slo=terminal_sli(finished,slo_cfg)
    alert=dual_window_alert(slo['events'],svc.now,slo_cfg)
    report={'scenario':scenarios,'telemetry':telemetry,
            'slo':slo,'alert_current_window':alert,
            'alert_synthetic_examples':alert_examples(),
            'diagnoses':diagnose(telemetry),
            'resource':{'live_blocks_after_flush':physical['live_blocks'],
                        'peak_blocks':physical['peak_live_blocks'],
                        'physical_kv_bytes':physical['physical_storage_bytes'],
                        'zeroized_releases':physical['zeroized_releases'],
                        'model_tokens_processed':svc.runner.runner.tokens_processed,
                        'generated_tokens':svc.total_tokens},
            'boundaries':['logical_ticks_not_milliseconds','single_process_real_CPU_KV',
                          'toy_random_LM_not_pretrained','custom_trace_schema_not_OTLP',
                          'in_memory_metrics_not_Prometheus_server','synthetic_alert_thresholds',
                          'hashed_correlation_ids_not_prod_secure_identity',
                          'no_network_no_distributed_context_propagation']}
    return report

### 实验编排与权重重载辅助函数

下面两个折叠代码单元定义诊断钻取、落盘和独立重放，供接下来的完整演练使用。它们仍在Notebook内，不读取外部脚本。

In [24]:
# %% 17 故障钻取：从低基数指标定位一条受保护的Trace，不读取原始文本

def trace_drilldown(report,terminal_status):
    require(terminal_status in StreamGateway.terminal_states,'需要允许的终态类型')
    traces=report['telemetry']['spans'];logs=report['telemetry']['structured_logs']
    roots=sorted((r for r in traces if r['parent_span_id'] is None and r['status']==terminal_status),
                 key=lambda x:x['trace_id'])
    require(bool(roots),'当前场景没有这种终态')
    root=roots[0]
    related=[s for s in traces if s['trace_id']==root['trace_id']]
    related_logs=[line for line in logs if line['trace_id']==root['trace_id']]
    require(all(x['trace_id']==root['trace_id'] for x in related+related_logs),'跨trace混合')
    return {'terminal':terminal_status,'trace_id':root['trace_id'],
            'model_version':root['attributes']['model_version'],
            'model_fingerprint':root['attributes']['model_fingerprint'],
            'root_span_id':root['span_id'],
            'span_names':[s['name'] for s in related],
            'log_events':[x['event'] for x in related_logs],
            'latency_ticks':root['end_tick']-root['start_tick'],
            'scope':'correlated_observations_not_verified_root_cause'}

In [25]:
# %% 18 模型权重的独立重建、实验保存与安全的离线检查

def save_models(path,tok,models,seed):
    blob={'version':VERSION,'seed':seed,
          'model_config':asdict(next(iter(models.values())).cfg),'tokenizer':tok.contract(),
          'hashes':{v:state_hash(m.state_dict()) for v,m in models.items()},
          'states':{v:{k:t.detach().cpu().clone() for k,t in m.state_dict().items()}
                    for v,m in models.items()}}
    torch.save(blob,path)


def load_models(path):
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'实验版本不匹配')
    tok,models=build_models(blob['seed'])
    require(blob['model_config']==asdict(next(iter(models.values())).cfg) and
            blob['tokenizer']==tok.contract(),'模型/词表身份不匹配')
    require(set(models)==set(blob['states'])==set(blob['hashes']),'adapter版本集合不匹配')
    for version,m in models.items():
        require(state_hash(blob['states'][version])==blob['hashes'][version],
                '模型权重摘要不匹配')
        m.load_state_dict(blob['states'][version],strict=True)
        require(state_hash(m.state_dict())==blob['hashes'][version],'恢复参数不一致')
        m.eval().requires_grad_(False)
    return tok,models,blob['seed']


def reload_run(folder):
    folder=Path(folder)
    tok,models,seed=load_models(folder/'models.pt')
    expected=json.loads((folder/'scenario.json').read_text(encoding='utf-8'))
    actual=real_service_scenarios(seed,model_bundle=(tok,models))
    require(actual==expected,'干净权重重放与场景/trace/metrics不一致')
    require(actual['resource']['live_blocks_after_flush']==0,'重放出现KV资源泄漏')
    return {'exact_full_report_equal':True,'model_hash_verified':True,
            'model_versions':sorted(models),'trace_roots':actual['telemetry']['reconciliation']['logical_requests'],
            'kv_live_after_flush':actual['resource']['live_blocks_after_flush'],
            'scope':'inference_scenario_replay_not_crash_recovery_or_OTLP_ingest'}


def fixture_demo(output_root=Path('lesson_78_outputs'),seed=7801):
    require(type(seed) is int and 0<=seed<=2**32-1,'seed必须为32位非负整数')
    seed_all(seed);torch.set_num_threads(1)
    folder=Path(output_root)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%fZ')
    folder.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'seed':seed,'python':platform.python_version(),
              'torch':str(torch.__version__),'device':'cpu','status':'running','sources':SOURCES,
              'limitations':['real_CPU_KV_not_GPU_measurement','logical_ticks_not_real_ms',
                'in_process_custom_traces_not_OTLP','in_memory_metrics_not_Prometheus_endpoint',
                'no_distributed_traceparent','single_process_no_crash_recovery',
                'teaching_hash_ids_not_production_authentication',
                'no_prompt_text_or_token_id_in_telemetry','synthetic_random_model_not_pretrained']}
    write_json(folder/'manifest.json',manifest)
    try:
        tok,models=build_models(seed)
        save_models(folder/'models.pt',tok,models,seed)
        report=real_service_scenarios(seed,model_bundle=(tok,models))
        write_json(folder/'scenario.json',report)
        write_json(folder/'traces.json',report['telemetry']['spans'])
        write_json(folder/'logs.json',report['telemetry']['structured_logs'])
        write_json(folder/'metrics.json',{'counters':report['telemetry']['metric_counters'],
                                         'histograms':report['telemetry']['metric_histograms']})
        write_json(folder/'slo.json',report['slo'])
        write_json(folder/'alerts.json',{'actual':report['alert_current_window'],
                                        'fixtures':report['alert_synthetic_examples']})
        write_json(folder/'diagnosis.json',report['diagnoses'])
        write_json(folder/'capacity_trace.json',trace_drilldown(report,'capacity_rejected'))
        write_json(folder/'sources.json',SOURCES)
        loaded=reload_run(folder)
        write_json(folder/'reload.json',loaded)
        manifest['status']='completed';write_json(folder/'manifest.json',manifest)
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(folder/'manifest.json',manifest)
        raise
    print('输出目录:',folder)
    print('唯一操作:',report['telemetry']['reconciliation']['logical_requests'],
          '提交尝试:',report['telemetry']['reconciliation']['attempts'])
    print('真实KV生成token:',report['resource']['generated_tokens'],
          '真实prefill计算token:',report['resource']['model_tokens_processed']-report['resource']['generated_tokens'])
    print('服务SLI:',report['slo']['good'],'/',report['slo']['eligible'],
          '（不把取消计入分母）')
    print('最后物理KV占用:',report['resource']['live_blocks_after_flush'])
    return folder

In [26]:
RUN_FULL_DEMO = True
OUTPUT_ROOT=Path('lesson_78_outputs')
RUN_FOLDER=None
if RUN_FULL_DEMO:
    RUN_FOLDER=fixture_demo(OUTPUT_ROOT,7801)
    live_report=json.loads((RUN_FOLDER/'scenario.json').read_text(encoding='utf-8'))
else:
    print('已跳过完整真实KV场景，不制造模拟执行结果。')

输出目录: lesson_78_outputs/run_20261105_051125_364590Z
唯一操作: 9 提交尝试: 13
真实KV生成token: 5 真实prefill计算token: 73
服务SLI: 1 / 4 （不把取消计入分母）
最后物理KV占用: 0


## 11. 观察真实结果：请求、attempt、delta、SLO与资源

本制作环境的默认CPU演练得到**9个逻辑操作、13次提交尝试**；重试与冲突不重复增加逻辑操作。物理模型完成5个生成token和73个新的prefill token，真实共享前缀命中了4个token。最终物理KV页为零。

完成分布为：1次 done、5次客户端取消、1次绝对超时、1次慢消费者、1次容量拒绝。按事先制定的SLI排除5次取消，因此eligible为4，good为1；这是一次刻意包含故障的演练，**25%只是演练内服务完成率，不是线上生产SLO或模型成功率**。

只有真实发布delta的2个逻辑操作记录了首次chunk tick；不能拿2作全部9个请求的TTFC分母。

In [27]:
if RUN_FOLDER is not None:
    x=live_report
    print('唯一操作 / 提交尝试:',x['telemetry']['reconciliation']['logical_requests'],
          '/',x['telemetry']['reconciliation']['attempts'])
    print('真实KV计算:',x['resource'])
    print('终态分布:',x['scenario']['terminal']['states'])
    print('首次chunk观测数:',x['telemetry']['reconciliation']['ttfc_observations'])
    print('SLO good / eligible:',x['slo']['good'],'/',x['slo']['eligible'])
    print('本场景最终窗口是否告警:',x['alert_current_window']['alert'])

唯一操作 / 提交尝试: 9 / 13
真实KV计算: {'generated_tokens': 5, 'live_blocks_after_flush': 0, 'model_tokens_processed': 78, 'peak_blocks': 15, 'physical_kv_bytes': 196608, 'zeroized_releases': 21}
终态分布: {'cancelled': 5, 'capacity_rejected': 1, 'deadline_exceeded': 1, 'done': 1, 'slow_consumer': 1}
首次chunk观测数: 2
SLO good / eligible: 1 / 4
本场景最终窗口是否告警: False


### 找到一条容量拒绝 Trace，而不打印用户文本

先选 `capacity_rejected` 类型，再获取其中一条 root 的模型版本、权重摘要、相关阶段与事件名。返回的数据不包含任何 prompt、token ID 或租户身份。

In [28]:
if RUN_FOLDER is not None:
    detail=trace_drilldown(live_report,'capacity_rejected')
    print('钻取结果:',json.dumps(detail,ensure_ascii=False,indent=2))
    print('隐私哨兵出现在telemetry里吗:',
          'PrivatePaymentTokenNeverLog123' in canonical(live_report['telemetry']))

钻取结果: {
  "terminal": "capacity_rejected",
  "trace_id": "472169749ca1c98bdbf94631686d62db",
  "model_version": "v0001",
  "model_fingerprint": "009e7f23c63826d0f8c7656f029334493a9b1450df2d6a1e60e93a1f04e0781d",
  "root_span_id": "a3efc0ccd4b80de6",
  "span_names": [
    "inference.request",
    "request.admission"
  ],
  "log_events": [
    "rejected",
    "terminal"
  ],
  "latency_ticks": 0,
  "scope": "correlated_observations_not_verified_root_cause"
}
隐私哨兵出现在telemetry里吗: False


### 各类告警的证据与局限

这些是指标支持的相关信号，不提供自动宣称的根因。若要将报告变成生产Runbook，需要更多上下文：真实网络TTFT、GPU allocator、队列统计、请求超时配置、采样率与部署变更时间线。

In [29]:
if RUN_FOLDER is not None:
    for item in live_report['diagnoses']:
        print(item['signal'],'证据数=',item['evidence_count'],item['interpretation'])

client_backpressure 证据数= 2 客户端确认窗口饱和；检查ack和消费速率
slow_consumer_terminal 证据数= 1 服务按租约关闭慢消费者；不等于模型推理错误
deadline_exceeded 证据数= 1 绝对截止时间触发；检查排队/慢消费/生成预算
admission_or_kv_capacity 证据数= 1 容量拒绝或KV不足；核对页池和并发


## 12. 保存与独立重放

默认生成本地目录（不覆盖已有）：

| 文件 | 含义 |
|---|---|
| `models.pt` | 两个冻结LoRA模型的完整权重、结构、词表与摘要 |
| `scenario.json` | 完整演练报告，含安全关联日志与采样分母 |
| `traces.json`、`logs.json` | 受限根span/子span及结构化日志 |
| `metrics.json` | 低基数 Counter 与逻辑tick直方图样本 |
| `slo.json`、`alerts.json`、`diagnosis.json` | 明确定义的SLI和教学告警/故障线索 |
| `capacity_trace.json`、`reload.json` | 一条诊断钻取和独立重载检查 |
| `checks.json`、`manifest.json` | 固定检查、环境与局限声明 |

模型检查点只从**自己生成或可信来源**读取；不是可继续运行的在线会话检查点。真正跨进程/进程崩溃后恢复，需要持久事务、上下文传播、事件保留策略和跨节点时钟，不是本课已经完成的功能。

In [30]:
if RUN_FOLDER is not None:
    print('独立构造权重并重演：',json.dumps(reload_run(RUN_FOLDER),ensure_ascii=False,indent=2))

独立构造权重并重演： {
  "exact_full_report_equal": true,
  "model_hash_verified": true,
  "model_versions": [
    "v0001",
    "v0002"
  ],
  "trace_roots": 9,
  "kv_live_after_flush": 0,
  "scope": "inference_scenario_replay_not_crash_recovery_or_OTLP_ingest"
}


## 13. 105项自动检查与故意错误检测

固定检查覆盖：百分位计算、Metric白名单、Trace父子关系与时间、拒绝隐私字段、服务完成SLI分母、双窗口告警、真实KV生成与命中、幂等重试计数、模型版本与权重身份、回收后无物理KV残留。

测试不会声称已跑真实 HTTP、OTel exporter、生产Prometheus或真实GPU延迟。GPU/多进程/时钟与日志保留限制是本课**必须保留的事实边界**。

In [31]:
# %% 15 固定检查、负向用例与回归测试
class Checks:
    def __init__(self): self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def close(self,name,a,b,tol=1e-10):
        if a is None or b is None:
            self.ok(name,a is b)
        else:
            self.ok(name,math.isclose(float(a),float(b),rel_tol=tol,abs_tol=tol))
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,AssertionError):
            self.names.append(name);return
        raise AssertionError(name+': 非法输入本应拒绝')


def run_checks():
    torch.set_num_threads(1)
    c=Checks()
    # 百分位：离散值使用nearest rank而非线性插值，明确N与时间单位。
    c.close('单样本p95',nearest_rank([2],.95),2)
    c.close('N=5的p95应为最大值',nearest_rank([4,1,3,2,8],.95),8)
    c.close('N=5的p50是第三项',nearest_rank([4,1,3,2,8],.5),3)
    c.close('N=4的p25为最小值',nearest_rank([9,3,1,2],.25),1)
    c.close('N=20的p95为第19项',nearest_rank(list(range(1,21)),.95),19)
    c.close('p100取最大值',nearest_rank([1,8,2],1),8)
    c.rejects('空窗口不捏造p95',lambda:nearest_rank([],.95))
    c.rejects('p0不是合法分位',lambda:nearest_rank([1],0))
    c.rejects('p大于1拒绝',lambda:nearest_rank([1],1.1))
    c.rejects('负值拒绝',lambda:nearest_rank([-1,2],.95))
    c.rejects('NaN拒绝',lambda:nearest_rank([float('nan')],.95))
    # Label低基数与固定单位
    metrics=SafeMetrics()
    metrics.count('service_requests_total',{'version':'v0001','result':'accepted'})
    metrics.count('service_requests_total',{'version':'v0001','result':'accepted'},2)
    c.close('Counter单调累加',metrics.count_value('service_requests_total',
        {'version':'v0001','result':'accepted'}),3)
    metrics.observe('service_latency_ticks',{'version':'v0002'},3)
    metrics.observe('service_latency_ticks',{'version':'v0002'},5)
    c.close('Histogram记录两个样本',metrics.distributions()[0]['count'],2)
    c.close('Histogram p95小样本为最大值',metrics.distributions()[0]['p95'],5)
    c.ok('指标单位仍是logical ticks',all(x['name'].endswith('_ticks') for x in metrics.distributions()))
    c.ok('指标审计通过',metrics.audit())
    c.rejects('指标禁止request ID维度',lambda:metrics.count('service_requests_total',
        {'version':'v0001','result':'accepted','request_id':'rs0001'}))
    c.rejects('指标禁止tenant标签',lambda:metrics.count('service_requests_total',
        {'version':'v0001','result':'accepted','tenant':'secret'}))
    c.rejects('指标禁止user_email',lambda:metrics.count('service_requests_total',
        {'version':'v0001','result':'accepted','email':'x@y.com'}))
    c.rejects('标签必须白名单值',lambda:metrics.count('service_requests_total',
        {'version':'v9999','result':'accepted'}))
    c.rejects('metric名称固定',lambda:metrics.count('custom_private_metric',{}))
    c.rejects('Counter不能负向更新',lambda:metrics.count('service_requests_total',
        {'version':'v0001','result':'accepted'},-1))
    c.rejects('Counter不能记录小数',lambda:metrics.count('service_requests_total',
        {'version':'v0001','result':'accepted'},.1))
    c.rejects('Histogram不能负时间',lambda:metrics.observe('service_latency_ticks',{'version':'v0001'},-2))
    c.rejects('Histogram不能NaN',lambda:metrics.observe('service_latency_ticks',{'version':'v0001'},float('nan')))
    c.rejects('Histogram也不能记录traceID标签',lambda:metrics.observe('service_latency_ticks',
        {'version':'v0001','trace_id':'foo'},2))
    # 结构化Trace：父子、关闭、白名单、用户文本排除。
    tr=SafeTraceLog(seed=7801);model_fp=digest('fixed_model');tpl_fp=digest('template')
    root=tr.open('rs0000',0,'v0001',model_fp,tpl_fp)
    c.ok('trace_id32位十六进制',bool(TRACE_ID_RE.fullmatch(root['trace_id'])))
    c.ok('span_id16位十六进制',bool(SPAN_ID_RE.fullmatch(root['span_id'])))
    c.ok('root没有父span',root['parent_span_id'] is None)
    child=tr.add_span('rs0000','inference.prefill',0,0,{'input_tokens':6})
    c.ok('child指向root',child['parent_span_id']==root['span_id'])
    c.ok('同一trace内父子关联',child['trace_id']==root['trace_id'])
    tr.log('rs0000','admitted',0,child,{'input_tokens':6})
    c.ok('log关联有效span',tr.logs[0]['span_id']==child['span_id'])
    tr.finish('rs0000',2,'done')
    c.close('root终态tick',root['end_tick'],2)
    c.ok('root状态由终态回填',root['status']=='done')
    c.ok('trace关闭后审计通过',tr.audit()['trace_roots']==1)
    c.rejects('重复关闭不允许',lambda:tr.finish('rs0000',3,'done'))
    c.rejects('禁止向已关闭root追加晚到span',lambda:tr.add_span('rs0000','stream.poll',3,3))
    tr.log('rs0000','retry',3,attrs={'new_compute':0})
    c.ok('晚到重试可关联日志但不新建过期子span',len(tr.spans)==2 and tr.logs[-1]['tick']==3)
    open_trace=SafeTraceLog(seed=7811)
    open_trace.open('rs0002',0,'v0001',digest('m'),digest('t'))
    c.rejects('活跃span拒绝原始prompt',lambda:open_trace.add_span(
        'rs0002','stream.poll',0,0,{'prompt':'TOP_SECRET'}))
    c.rejects('日志拒绝原始token_id',lambda:tr.log('rs0000','token_committed',3,attrs={'token_id':17}))
    c.rejects('trace不能接收高基数未知属性',lambda:tr.log('rs0000','admitted',3,attrs={'user_id':100}))
    c.rejects('model指纹必须hash',lambda:tr.log('rs0000','admitted',3,attrs={'model_fingerprint':'secret'}))
    c.rejects('负持续时间拒绝',lambda:tr.add_span('rs0000','stream.poll',2,1))
    c.rejects('没有root不能添加子span',lambda:tr.add_span('rs9999','stream.poll',1,1))
    c.rejects('未知日志事件拒绝',lambda:tr.log('rs0000','hello_private',3))
    altered=copy.deepcopy(tr);altered.spans[1]['parent_span_id']='0'*16
    c.rejects('悬空父span会被审计发现',lambda:altered.audit())
    altered=copy.deepcopy(tr);altered.spans[1]['trace_id']='0'*32
    c.rejects('跨请求trace串线被发现',lambda:altered.audit())
    altered=copy.deepcopy(tr);altered.spans[1]['start_tick']=-1
    c.rejects('负tick被审计拒绝',lambda:altered.audit())
    altered=copy.deepcopy(tr);altered.spans[1]['attributes']['prompt']='raw'
    c.rejects('追加用户内容后审计失败',lambda:altered.audit())
    altered=copy.deepcopy(tr);altered.spans.append(altered.spans[1])
    c.rejects('重复span ID被审计拒绝',lambda:altered.audit())
    # 显式SLI：服务完成、取消排除、拒绝计入、边界、零分母。
    config=SLOConfig()
    rows=[{'accepted_tick':0,'end_tick':8,'state':'done'},
          {'accepted_tick':0,'end_tick':9,'state':'done'},
          {'accepted_tick':1,'end_tick':1,'state':'capacity_rejected'},
          {'accepted_tick':1,'end_tick':4,'state':'cancelled'}]
    sli=terminal_sli(rows,config)
    c.close('SLO offered不是attempt数',sli['offered'],4)
    c.close('SLO分母排除主动取消',sli['eligible'],3)
    c.close('SLO好事件包含截止边界',sli['good'],1)
    c.close('SLO拒绝与慢完成都算bad',sli['bad'],2)
    c.close('SLO比例分母按eligible',sli['good_fraction'],1/3)
    c.close('服务done延迟p95使用nearest rank',sli['p95_done_latency_ticks'],9)
    cancelled_only=terminal_sli([rows[-1]],config)
    c.ok('全部主动取消时SLI无分母',cancelled_only['good_fraction'] is None)
    c.rejects('不能把非终态送入SLI',lambda:terminal_sli(
        [{'accepted_tick':0,'end_tick':1,'state':'active'}],config))
    c.rejects('SLI禁止携带用户内容',lambda:terminal_sli([dict(rows[0],prompt='hi')],config))
    c.rejects('SLI拒绝负耗时',lambda:terminal_sli(
        [{'accepted_tick':2,'end_tick':1,'state':'done'}],config))
    # 教学双窗口体量与burn门控：持续坏、健康、低流量不误报。
    examples=alert_examples()
    c.ok('持续错误触发告警',examples['sustained']['alert'])
    c.ok('健康窗口不触发告警',not examples['healthy']['alert'])
    c.ok('样本不足不触发告警',not examples['low_volume']['alert'])
    c.close('长窗口持续故障样本数',examples['sustained']['long']['eligible'],14)
    c.close('短窗口持续故障样本数',examples['sustained']['short']['eligible'],5)
    c.close('持续故障长窗bad比例',examples['sustained']['long']['bad_fraction'],.5)
    c.close('持续故障短窗bad比例',examples['sustained']['short']['bad_fraction'],1.)
    c.close('burn率除以错误预算',examples['sustained']['long']['burn_rate'],5.)
    c.ok('不够样本必须标出volume不足',not examples['low_volume']['short']['sufficient_volume'])
    # 防止把两个窗口误写成OR：故意制造只有单侧越限的两个场景。
    late_single=[{'end_tick':i,'eligible':True,'good':i!=14,'bad':i==14}
                 for i in range(1,15)]
    early_many=[{'end_tick':i,'eligible':True,'good':i>9,'bad':i<=9}
                for i in range(1,15)]
    c.ok('只有短窗越限不告警',not dual_window_alert(late_single,14,config)['alert'])
    c.ok('只有长窗越限不告警',not dual_window_alert(early_many,14,config)['alert'])
    c.rejects('告警规则不允许非法窗口顺序',lambda:SLOConfig(short_window_ticks=20,long_window_ticks=10).validate())
    c.rejects('告警规则不允许target1',lambda:SLOConfig(target=1.).validate())
    c.rejects('非法alert输入不能附加raw字段',lambda:dual_window_alert(
        [dict(examples['healthy']['short'],email='secret')],10,config))
    c.rejects('告警输入不能标成同时good和bad',lambda:dual_window_alert(
        [{'end_tick':1,'eligible':True,'good':True,'bad':True}],1,config))
    c.ok('空系统诊断不捏造根因',diagnose({'metric_counters':[]})[0]['signal']=='no_known_fault_signal')
    # 真实因果模型＋真实分页KV：可复跑，模型身份与请求关联。
    report=real_service_scenarios(7801)
    tel=report['telemetry'];recon=tel['reconciliation']
    c.close('真实模型请求数',recon['logical_requests'],9)
    c.close('真实服务终态数',recon['terminal'],9)
    c.close('重试按attempt分开计数',recon['attempts'],13)
    c.close('模型生成token与counter守恒',recon['generated_tokens'],report['resource']['generated_tokens'])
    c.close('仅真实delta计入TTFC',recon['ttfc_observations'],2)
    c.ok('同请求重试不多建root',report['scenario']['idempotency']['same_trace_for_retries'])
    c.ok('重试并不重复计算',report['scenario']['ttl']['expired_no_recompute'])
    c.ok('会话在真实模型上确实有共享前缀命中',report['scenario']['prefill']['shared_prefix_hit_tokens']>0)
    c.close('模型版本在trace中保留',report['scenario']['prefill']['unique_models'],2)
    c.ok('全流程有backpressure证据',report['scenario']['terminal']['backpressure_ticks']>0)
    c.ok('有有效的deadline事件',report['scenario']['terminal']['states'].get('deadline_exceeded')==1)
    c.ok('有有效的容量拒绝',report['scenario']['terminal']['states'].get('capacity_rejected')==1)
    c.ok('有慢消费者关闭',report['scenario']['terminal']['states'].get('slow_consumer')==1)
    c.ok('包含模型版本与真实权重指纹',all('model_fingerprint' in x['attributes']
                 for x in tel['spans'] if x['name']=='inference.request'))
    c.close('Span root数量与请求守恒',sum(x['name']=='inference.request' for x in tel['spans']),9)
    c.ok('有prefill和decode子span',{'inference.prefill','inference.decode'}<=set(x['name'] for x in tel['spans']))
    c.ok('没有超出根span时段的子span',all(x['end_tick'] is not None
        for x in tel['spans']))
    c.ok('trace和日志可安全JSON编码',bool(canonical(tel)))
    c.ok('遥测无用户文本', 'PrivatePaymentTokenNeverLog123' not in canonical(tel))
    c.ok('遥测无原始token ID字段','"token_id"' not in canonical(tel))
    c.ok('指标不含trace维度',all(set(x['labels'])<= {'version','reason','result','kind'}
                                           for x in tel['metric_counters']))
    c.ok('高基数trace只出现在trace与日志里',all('trace_id' not in x['labels']
                                           for x in tel['metric_counters']))
    c.ok('真实KV完全释放',report['resource']['live_blocks_after_flush']==0)
    c.ok('终态资源回收记录存在',any(x['name']=='kv.release' for x in tel['spans']))
    c.close('提供的SLI分母可审计',report['slo']['eligible'],4)
    c.close('客户端取消的SLI排除项',report['slo']['excluded_client_cancellations'],5)
    c.ok('诊断全都标为correlated而非proved',all(x['certainty']=='correlated_signal_not_proven_root_cause'
                                       for x in report['diagnoses']))
    c.ok('真实场景有四类相关故障信号',len(report['diagnoses'])==4)
    c.ok('模拟短窗高错误和小体量不等于真实时延告警',not report['alert_current_window']['alert'])
    c.ok('报告声明逻辑时钟','logical_ticks_not_milliseconds' in report['boundaries'])
    c.ok('报告明确没有生产OTLP接入','custom_trace_schema_not_OTLP' in report['boundaries'])
    return {'passed':len(c.names),'names':c.names,
            'scope':'单进程CPU真实KV与逻辑tick；不代表生产级OTLP/网络/毫秒SLO'}

In [32]:
check_report=run_checks()
print('固定检查通过:',check_report['passed'],'项')
print('检查范围:',check_report['scope'])
if RUN_FOLDER is not None:write_json(RUN_FOLDER/'checks.json',check_report)

固定检查通过: 107 项
检查范围: 单进程CPU真实KV与逻辑tick；不代表生产级OTLP/网络/毫秒SLO


## 14. 独立练习：默认关闭，先写再看答案

### 练习A：低基数指标标签

编写 `safe_labels_ex(name,labels)`，返回排序后的键值元组。要求指标名称和标签键集合严格匹配声明，而且值在允许列表里。任何额外 request ID、用户邮箱、未知模型版本都抛出 `ValueError`。

*提示：你已经有本课注册表 `METRIC_LABEL_SPECS`，本练习自己写校验，不直接转调完整答案。*

In [33]:
def exercise_a(fn):
    """写出严格的低基数label验证函数；返回排序tuple，拒绝未知维度。"""
    good={'version':'v0001','result':'accepted'}
    value=fn('service_requests_total',good)
    require(value==(('result','accepted'),('version','v0001')),'合法label排序错误')
    for name,labels in [
        ('service_requests_total',{'version':'v0001','result':'accepted','request_id':'id'}),
        ('service_requests_total',{'version':'vX','result':'accepted'}),
        ('unregistered',{}),
        ('service_requests_total',{'version':'v0001'}),
    ]:
        try:fn(name,labels)
        except ValueError:continue
        raise LessonError('练习A未拒绝危险label')
    return {'exercise':'strict_metric_labels','passed':5}


def exercise_b(fn):
    """实现nearest rank百分位，不能隐式插值或凭空生成空分位。"""
    require(fn([8,2,1,4,3],.95)==8,'p95位置错误')
    require(fn([8,2,1,4,3],.5)==3,'p50位置错误')
    require(fn(list(range(1,21)),.95)==19,'20点的p95应是第19项')
    for values,p in [([],.5),([1],0),([1],1.1)]:
        try:fn(values,p)
        except ValueError:continue
        raise LessonError('练习B未拒绝非法输入')
    return {'exercise':'nearest_rank','passed':6}


def exercise_c(fn):
    """两个窗口均需有足量、均超burn阈值才告警。"""
    cfg=SLOConfig()
    examples=alert_examples()
    require(fn(examples['sustained']['short'],examples['sustained']['long'],cfg),'持续异常不报警')
    require(not fn(examples['healthy']['short'],examples['healthy']['long'],cfg),'健康场景误报')
    require(not fn(examples['low_volume']['short'],examples['low_volume']['long'],cfg),'小流量误报')
    lower=copy.deepcopy(examples['sustained']['long']);lower['sufficient_volume']=False
    require(not fn(examples['sustained']['short'],lower,cfg),'长窗口体量不足误报')
    lower=copy.deepcopy(examples['sustained']['short']);lower['burn_rate']=0.
    require(not fn(lower,examples['sustained']['long'],cfg),'短窗不达阈值误报')
    return {'exercise':'dual_window_gate','passed':5}

In [34]:
def safe_labels_ex(name,labels):
    raise NotImplementedError('请实现严格指标标签验证')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:print(exercise_a(safe_labels_ex))

### 练习B：Nearest-rank 百分位

编写 `percentile_ex(values,p)`：输入非空有限非负值列表，`p`在`(0,1]`，使用 `sorted(values)[ceil(p*N)-1]`。无效数据必须显式拒绝，不能默认填0；不要在本练习中使用线性插值。

In [35]:
def percentile_ex(values,p):
    raise NotImplementedError('请独立实现nearest-rank')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:print(exercise_b(percentile_ex))

### 练习C：双窗口告警门控

编写 `dual_window_gate_ex(short,long,cfg)`：**两**个窗口体量必须达标，**两**个窗口burn_rate都达到阈值才返回True。不能写成 `or`；低体量下高坏率不等于本课规定的立即告警。

In [36]:
def dual_window_gate_ex(short,long,cfg):
    raise NotImplementedError('请独立实现双窗口门控')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:print(exercise_c(dual_window_gate_ex))

## 15. 参考答案

只有在完成练习后再展开执行。下列答案名字独立，不覆盖空白函数。

In [37]:
def answer_a(name,labels):
    return strict_labels(name,labels,METRIC_LABEL_SPECS)


def answer_b(values,p):
    if type(values) is not list or not values:raise ValueError('需要非空列表')
    if type(p) not in (int,float) or not math.isfinite(p) or not 0<p<=1:
        raise ValueError('分位点必须在(0,1]')
    if not all(type(x) in (int,float) and math.isfinite(x) and x>=0 for x in values):
        raise ValueError('数值非法')
    ordered=sorted(values)
    return ordered[math.ceil(p*len(ordered))-1]


def answer_c(short_window,long_window,cfg):
    cfg.validate()
    for x in (short_window,long_window):
        if not (type(x) is dict and type(x.get('sufficient_volume')) is bool
                and type(x.get('burn_rate')) in (int,float)):
            raise ValueError('短长窗口字段不合法')
    return bool(short_window['sufficient_volume'] and long_window['sufficient_volume']
                and short_window['burn_rate']>=cfg.burn_threshold
                and long_window['burn_rate']>=cfg.burn_threshold)

In [38]:
print('A:',exercise_a(answer_a))
print('B:',exercise_b(answer_b))
print('C:',exercise_c(answer_c))

A: {'exercise': 'strict_metric_labels', 'passed': 5}
B: {'exercise': 'nearest_rank', 'passed': 6}
C: {'exercise': 'dual_window_gate', 'passed': 5}


## 16. 阶段验收

1. 能解释一个服务操作为什么对应1个root，但会有多个日志和重试attempt。
2. 明白为什么请求ID可放Trace关联，却不能直接作为 Prometheus 指标 label。
3. 在用户prompt含敏感值时，安全结构化日志为什么应该采用字段白名单。
4. 能手算 `offered/eligible/good`，说清取消、容量拒绝、deadline的分母规则。
5. 能区分 `TTFC` 的观测数量、逻辑 tick、真实系统毫秒以及小样本p95。
6. 知道告警关联信号不能单独证明根因；跨节点/崩溃恢复需要额外保证。

**下一课：第79课——推理服务故障演练、Runbook 与自动回滚策略。** 把 Trace、指标、事件与版本发布控制接成一条可审计的应急处置流程。

## 参考资料

- [OpenTelemetry：Traces](https://opentelemetry.io/docs/concepts/signals/traces/) 与 [Context propagation](https://opentelemetry.io/docs/concepts/context-propagation/)
- [Prometheus：Metric 与 Label 命名、高基数风险](https://prometheus.io/docs/practices/naming/)
- [Google SRE：Alerting on SLOs，双窗口与低流量](https://sre.google/workbook/alerting-on-slos/)
- [OpenTelemetry GenAI 语义规范](https://opentelemetry.io/docs/specs/semconv/gen-ai/)（持续演进；本课未声称实现其完整字段）
- [MDN：SSE流](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events)
- [vLLM前缀缓存身份与安全边界](https://docs.vllm.ai/en/latest/design/prefix_caching/)

所有文献只是概念与实现取舍的参考；本课得到的计数是本地真实KV＋模拟事件时钟的教学结果，不能引用为这些项目的真实性能数据。